In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from shapely.geometry import mapping
import rasterio
from rasterio.enums import Resampling

高程数据：https://github.com/globalmaps/gm_el_v1; 

坡度根据高程，利用ArcGIS pro的xxx功能，计算1km网格，并重采样至0.25度网格。

土地利用数据：https://cds.climate.copernicus.eu/datasets/satellite-land-cover?tab=download

风速、光照、温度数据：https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download

保护区数据：https://www.protectedplanet.net/en 下载每个大洲的数据，利用ArcGIS PRO进行图层拼接，并将原始的shp文件进行栅格化成1km的栅格格点，并重采样至0.25度网格。

雪覆盖：https://zenodo.org/records/6011200

## workflow
+ 计算各省CF
$$ CF_{i,t} = \frac{\sum_j{CF\cdot C}}{\sum_{j}{C}} $$
    + 评估装机潜力，与土地类型和坡度有关
    + 评估CF，风速决定

In [ ]:
# 打开包含风速、温度和太阳辐射的 NetCDF 文件
# 文件名: "6windtempsolar201920.nc"
slope = xr.open_dataset(r"F:\Workspace\DATA_DEM\Slope_map_1km.tif")["band_data"][0, :, :] #作为投影参考
data = xr.open_dataset(r"E:\Workspace\2023.nc")

# 提取 100 米高度的东向风速 (u 分量) 和北向风速 (v 分量)
# 这里使用 8760 索引开始，表示从第二年的数据开始（假设每小时记录一次）
i=0
u100 = data["u100"][0:8760, :, :]
v100 = data["v100"][0:8760, :, :]
u100 = u100.rio.write_crs(slope.rio.crs) #重新进行投影
v100 = v100.rio.write_crs(slope.rio.crs) #重新进行投影

In [ ]:
slope_025d = xr.open_dataset(r"F:\Workspace\DATA_DEM\Slope_map_025d.tif")["band_data"][0, :, :]
slope_1km = xr.open_dataset(r"F:\Workspace\DATA_DEM\Slope_map_1km.tif")["band_data"][0, :, :]
lucc_025d = xr.open_dataset(r"F:\Workspace\DATA_Landuse\World_landuse_025d.tif")["band_data"][0, :, :]
lucc_1km = xr.open_dataset(r"F:\Workspace\DATA_Landuse\World_landuse_1km.tif")["band_data"][0, :, :]
elevation_025d = xr.open_dataset(r"F:\Workspace\DATA_DEM\merged_global_map_025d.tif")["band_data"][0, :, :]
elevation_1km = xr.open_dataset(r"F:\Workspace\DATA_DEM\merged_global_map.tif")["band_data"][0, :, :]
protected_025d = xr.open_dataset(r"F:\Workspace\DATA_Protected\Protected_025d.tif")["band_data"][0, :, :].rio.reproject_match(u100, resampling=Resampling.mode)
protected_1km = xr.open_dataset(r"F:\Workspace\DATA_Protected\Protected_1km.tif")["band_data"][0, :, :]
classification_df = pd.read_csv("land_use_classification.csv")
lucc_Wind_weights = dict(zip(classification_df['Flag Value'], classification_df['Onshore wind-Base']))
lucc_Solar_weights = dict(zip(classification_df['Flag Value'], classification_df['Solar-Base']))
lucc_cf = dict(zip(classification_df['Flag Value'], classification_df['CF']))
lucc_cf_noWater = dict(zip(classification_df['Flag Value'], classification_df['CF_noWater']))
slope_1km = slope_1km.rename({'y': 'latitude', 'x': 'longitude'})

In [ ]:
# --- 1) LULC → 权重映射函数（保持不变） ---
def lucc_wind_process_weighted(x):
    return lucc_Wind_weights.get(x, np.nan)

def lucc_solar_process_weighted(x):
    return lucc_Solar_weights.get(x, np.nan)

def lucc_cf_noWater_weighted(x):
    return lucc_cf_noWater.get(x, np.nan)

def lucc_cf_weighted(x):
    return lucc_cf.get(x, np.nan)

# --- 2) 生成 0.25° 与 1 km 的 LULC 权重（名称更清晰） ---
lucc_wind_025d        = lucc_025d.map_blocks(np.vectorize(lucc_wind_process_weighted))
lucc_solar_025d       = lucc_025d.map_blocks(np.vectorize(lucc_solar_process_weighted))
lucc_cf_nowater_025d  = lucc_025d.map_blocks(np.vectorize(lucc_cf_noWater_weighted))
lucc_cf_025d          = lucc_025d.map_blocks(np.vectorize(lucc_cf_weighted))

lucc_wind_1km         = lucc_1km.map_blocks(np.vectorize(lucc_wind_process_weighted))
lucc_solar_1km        = lucc_1km.map_blocks(np.vectorize(lucc_solar_process_weighted))
lucc_cf_nowater_1km   = lucc_1km.map_blocks(np.vectorize(lucc_cf_noWater_weighted))
lucc_cf_1km           = lucc_1km.map_blocks(np.vectorize(lucc_cf_weighted))  # 修正了原来的拼写 ucc_cf_1km

# === 关键：为“1 km 主计算”提供一致的变量名 ===
# 风电主线：把风电用的 LULC 权重直接命名为 `lucc`
lucc = lucc_wind_1km            # (latitude, longitude) ∈ [0,1]

# 如需做光伏主线，保留一个 `lucc_solar` 供后续 PV 计算使用
lucc_solar = lucc_solar_1km

# --- 3) 坡度 / 高程 / 保护地处理（输出名统一） ---
def Wind_slope_process(x):
    # 开发阈值：0–3° 可建，其余不可建（“且 &”而不是“或 |”）
    # 可选：放开/收紧时，替换数值即可
    conditions = [(x >= 0) & (x <= 3), (x > 3)]
    vals = [1, 0]
    return np.select(conditions, vals, default=np.nan)

def Solar_slope_process(x):
    # 基准：0–5° 可建
    conditions = [(x >= 0) & (x <= 5), (x > 5)]
    vals = [1, 0]
    return np.select(conditions, vals, default=np.nan)

def elevation_process(x):
    # ≤3000 m 可建
    conditions = [x <= 3000, x > 3000]
    vals = [1, 0]
    return np.select(conditions, vals, default=np.nan)

def protected_process(x):
    # 0=非保护地 可建；>0=保护地 不可建
    conditions = [x == 0, x > 0]
    vals = [1, 0]
    return np.select(conditions, vals, default=np.nan)

# —— 统一后的 1 km 变量名（与上一轮主计算对齐）——
wind_slope = Wind_slope_process(slope_1km)       # for wind pipeline
solar_slope = Solar_slope_process(slope_1km)     # for PV pipeline（若需要）
elevation  = elevation_process(elevation_1km)
protected  = protected_process(protected_1km)

# 如需要保留 0.25° 的对照版本，也给出同名后缀
wind_slope_025d = Wind_slope_process(slope_025d)
solar_slope_025d = Solar_slope_process(slope_025d)
elevation_025d = elevation_process(elevation_025d)
protected_025d = protected_process(protected_025d)

# --- 4) 风速/辐照到 CF 的函数（名称保留，便于单独调用） ---
def CF_process_wind(x):
    # 例：基于风速（m/s）的分段 CF 近似
    conditions = [(x < 3) | (x > 25), (x < 5), (x < 12), (x <= 25)]
    CFs = [0, 125*(x + 1e-7 - 3)/2500, (250 + 2250/7 * (x - 5))/2500, 1]
    return np.select(conditions, CFs, default=np.nan)

def pv_power(T, S, wind10):
    # 温度影响 + 系统效率，返回小时 CF（无量纲）
    T_p = 4.3 + 0.943 * T + 0.028 * S / 3600 - 1.528 * wind10
    TEM = (1 - 0.005 * (T_p - 25))
    SYS = 0.85
    return TEM * SYS

def pv_power1(T, S, wind10):
    # 带入辐照量换算到 kW·h 的形式（仍返回无量纲 CF 近似）
    T_p = 4.3 + 0.943 * T + 0.028 * S / 3600 - 1.528 * wind10
    TEM = (1 - 0.005 * (T_p - 25))
    SYS = 0.85
    return S / 1000 / 3600 * TEM * SYS


In [ ]:
import glob
import numpy as np
import xarray as xr

# ===== 参数（与你之前一致）=====
turbine_rated_kw = 2500
rotor_diam_km    = 0.103
spacing_multiple = 8
avail_factor     = 0.9

# ===== 工具函数 =====
def standardize_latlon_names_da(da: xr.DataArray) -> xr.DataArray:
    """把 DataArray 的 (lat/lon | y/x) 统一为 latitude/longitude（坐标+维度一起重命名）"""
    if not isinstance(da, xr.DataArray):
        raise TypeError("standardize_latlon_names_da 仅接受 DataArray")
    rename_dims = {}
    if "lat" in da.dims: rename_dims["lat"] = "latitude"
    if "lon" in da.dims: rename_dims["lon"] = "longitude"
    if "y"   in da.dims: rename_dims["y"]   = "latitude"
    if "x"   in da.dims: rename_dims["x"]   = "longitude"
    da = da.rename(rename_dims)

    rename_coords = {}
    if "lat" in da.coords: rename_coords["lat"] = "latitude"
    if "lon" in da.coords: rename_coords["lon"] = "longitude"
    if "y"   in da.coords: rename_coords["y"]   = "latitude"
    if "x"   in da.coords: rename_coords["x"]   = "longitude"
    if rename_coords:
        da = da.rename_vars(rename_coords).rename(rename_coords)
    return da

def pick_template(*cands):
    """
    从候选里挑一个带经纬度坐标的 DataArray 作为模板。
    常用地传入：lucc_1km, slope_1km, elevation_1km, protected_1km（原始 1km 数据）
    """
    for c in cands:
        if isinstance(c, xr.DataArray):
            da = standardize_latlon_names_da(c)
            if "latitude" in da.coords and "longitude" in da.coords:
                return da
    raise ValueError("未找到可用模板：请提供任一 1km 源 DataArray（如 lucc_1km/slope_1km 等）。")

def wrap_to_da(var, template: xr.DataArray, name: str) -> xr.DataArray:
    """
    把 numpy 或 DataArray 变成与 template 同网格/坐标的 DataArray。
    - 若为 DataArray：统一经纬度命名后，重采样到模板索引（仅对齐索引，不插值）。
    - 若为 numpy：直接用模板的 dims/coords 包装。
    """
    if isinstance(var, xr.DataArray):
        var = standardize_latlon_names_da(var)
        # 用 reindex_like 对齐索引（不改变值，仅按索引对齐）
        try:
            var = var.reindex_like(template, method=None, copy=False)
        except Exception:
            pass
        return var.rename(name)
    else:
        return xr.DataArray(var, dims=template.dims, coords=template.coords, name=name)

def align_longitudes(arr_like: xr.DataArray | xr.Dataset,
                     to_match: xr.DataArray | xr.Dataset):
    """把经度改成与 to_match 一致（0–360 或 -180–180），并按升序排序"""
    ref_lon = to_match["longitude"]
    to_0360 = (ref_lon.min() >= 0) and (ref_lon.max() <= 360)

    def _wrap(lon_vals, zero_to_360):
        lon = (lon_vals % 360 + 360) % 360
        if zero_to_360:
            return lon
        else:
            return xr.where(lon > 180, lon - 360, lon)

    obj = arr_like
    new_lon = _wrap(obj["longitude"], to_0360)
    sorter = np.argsort(new_lon.values)
    obj = obj.assign_coords(longitude=new_lon).isel(longitude=sorter)
    return obj

def cell_area_km2_from_coords(lat_1d: xr.DataArray, lon_1d: xr.DataArray) -> xr.DataArray:
    """规则经纬网像元面积（km²），A = R² Δλ (sinφ2 - sinφ1)"""
    R = 6371.0
    dlat = float(np.abs(np.diff(lat_1d.values)).mean())
    dlon = float(np.abs(np.diff(lon_1d.values)).mean())
    dlat_rad = np.deg2rad(dlat); dlon_rad = np.deg2rad(dlon)
    lat_ctr = np.deg2rad(lat_1d.values)
    lat_hi  = lat_ctr + 0.5 * dlat_rad
    lat_lo  = lat_ctr - 0.5 * dlat_rad
    strip = (R**2) * (np.sin(lat_hi) - np.sin(lat_lo))
    area_1d = strip * dlon_rad
    return xr.DataArray(
        np.repeat(area_1d[:, None], len(lon_1d), axis=1),
        dims=["latitude", "longitude"],
        coords={"latitude": lat_1d, "longitude": lon_1d},
        name="cell_area_km2", attrs={"units": "km^2"}
    )

def get_time_dim(da: xr.DataArray) -> str:
    for cand in ("valid_time", "time", "hour", "t"):
        if cand in da.dims:
            return cand
    raise KeyError(f"未找到时间维，现有维度：{da.dims}")

# ====== 1) 用你已有的 1km 源数据挑一个模板（至少要有一个是 DataArray）======
# 传入你命名空间里的变量：lucc_1km / slope_1km / elevation_1km / protected_1km 等
template = pick_template(
    globals().get("lucc_1km", None),
    globals().get("slope_1km", None),
    globals().get("elevation_1km", None),
    globals().get("protected_1km", None),
)

# ====== 2) 用模板把你这段代码算出的变量（可能是 numpy）包回 DataArray ======
# 这里直接用你已经得到的 lucc / wind_slope / elevation / protected
lucc_da       = wrap_to_da(globals()["lucc"],        template, "lucc")
wind_slope_da = wrap_to_da(globals()["wind_slope"],  template, "wind_slope")
elevation_da  = wrap_to_da(globals()["elevation"],   template, "elevation")
protected_da  = wrap_to_da(globals()["protected"],   template, "protected")

# 1km 全局坐标
lat_1km_all = template["latitude"]
lon_1km_all = template["longitude"]

# ====== 3) 主循环：把 0.25° CF 插值到 1 km，并用你提供的输入计算潜力 ======
GE_wind_list = []
CP_wind_list = []

for seg_path in sorted(glob.glob(r"D:\Workspace\average_2023_segment_*.nc")):
    ds = xr.open_dataset(seg_path, engine="netcdf4")
    # 读取 CF，并统一经纬度维名
    if isinstance(ds, xr.Dataset):
        # 变量名如不同请改这里
        CF = ds["CF_wind_avg"]
    else:
        CF = ds
    # 若是 Dataset 的变量，标准化维名
    if isinstance(CF, xr.DataArray):
        # 尝试把 lat/lon 统一为 latitude/longitude
        CF = CF.rename({k: v for k, v in {
            "lat": "latitude", "lon": "longitude",
            "y": "latitude", "x": "longitude"
        }.items() if k in CF.dims or k in CF.coords})
        ds = CF.to_dataset(name="CF_wind_avg")
    else:
        ds = standardize_latlon_names_da(ds["CF_wind_avg"]).to_dataset(name="CF_wind_avg")

    ds = align_longitudes(ds, template)
    CF  = ds["CF_wind_avg"]
    tdim = get_time_dim(CF)

    # 先时间求和，降成 2D（省内存）
    CF_sum_025d = CF.isel({tdim: slice(0, 8760)}).sum(dim=tdim)

    # 用当前段的经度范围裁剪 1km 网格
    eps = 1e-9
    lon_target = lon_1km_all.sel(longitude=slice(float(ds.longitude.min()) - eps,
                                                 float(ds.longitude.max()) + eps))
    lat_target = lat_1km_all

    # ∑CF 从 0.25° → 1 km 插值（如想更保守可改 method="nearest"）
    CF_sum = CF_sum_025d.interp(latitude=lat_target, longitude=lon_target, method="linear")

    # 同步裁剪你提供的 1 km 输入
    lucc_chunk       = lucc_da.sel(latitude=lat_target,       longitude=lon_target)
    wind_slope_chunk = wind_slope_da.sel(latitude=lat_target, longitude=lon_target)
    elevation_chunk  = elevation_da.sel(latitude=lat_target,  longitude=lon_target)
    protected_chunk  = protected_da.sel(latitude=lat_target,  longitude=lon_target)

    # 面积
    cell_area_km2 = cell_area_km2_from_coords(lat_target, lon_target)

    # 可建设掩膜（0~1）
    buildable = lucc_chunk * wind_slope_chunk * elevation_chunk * protected_chunk

    # 机位密度与装机
    spacing_km = spacing_multiple * rotor_diam_km
    per_turbine_area_km2 = spacing_km ** 2
    n_turbines = (cell_area_km2 / per_turbine_area_km2) * buildable

    cap_kW  = (n_turbines * turbine_rated_kw * avail_factor).astype("float32")
    CP_wind = (cap_kW * 1e-6).rename("CP_wind")  # kW → GW

    # 年发电量：kW × (∑CF) × h / 1e12 = PWh
    GE_wind = (cap_kW * CF_sum * 1e-12).rename("GE_wind")

    GE_wind_list.append(GE_wind)
    CP_wind_list.append(CP_wind)

# 沿经度拼接
GE_wind_all = xr.concat(GE_wind_list, dim="longitude")
CP_wind_all = xr.concat(CP_wind_list, dim="longitude")

# De-dup longitude seams from segmented concat
GE_wind_all = GE_wind_all.groupby("longitude").mean()
CP_wind_all = CP_wind_all.groupby("longitude").mean()


GE_wind_all = GE_wind_all.sortby(['latitude', 'longitude'])
CP_wind_all = CP_wind_all.sortby(['latitude', 'longitude'])

# （可选）保存
GE_wind_all.to_netcdf(r"D:\Workspace\GE_wind_1km_PWh.nc")
CP_wind_all.to_netcdf(r"D:\Workspace\CP_wind_1km_GW.nc")


In [ ]:
# Downsample the data for visualization and replace NaN with 0
downsample_factor = 10  # Adjust this value based on your needs
CP_wind_downsampled = CP_wind_all[::downsample_factor, ::downsample_factor]

# Replace NaN values with 0
CP_wind_downsampled = CP_wind_downsampled.fillna(0)

# 排序坐标以确保按递增顺序
CP_wind_downsampled = CP_wind_downsampled.sortby(['latitude', 'longitude'])

# 先检查数据的基本信息
print(f"CP_wind_downsampled shape: {CP_wind_downsampled.shape}")
print(f"Data type: {CP_wind_downsampled.dtype}")
print(f"Min value: {CP_wind_downsampled.min().values}")
print(f"Max value: {CP_wind_downsampled.max().values}")
print(f"Has NaN values: {CP_wind_downsampled.isnull().any().values}")
print(f"Non-zero values count: {(CP_wind_downsampled > 0).sum().values}")

# 检查坐标顺序
print(f"Latitude range: {CP_wind_downsampled.latitude.min().values} to {CP_wind_downsampled.latitude.max().values}")
print(f"Longitude range: {CP_wind_downsampled.longitude.min().values} to {CP_wind_downsampled.longitude.max().values}")

plt.figure(figsize=(15, 10))
try:
    CP_wind_downsampled.plot(cmap='viridis', robust=True)
    plt.title('Wind Power Capacity Potential (Downsampled)')
    plt.show()
except Exception as e:
    print(f"xarray plot failed: {e}")
    # 备用方案：使用matplotlib直接绘图
    plt.imshow(CP_wind_downsampled.values, cmap='viridis', aspect='auto', origin='lower')
    plt.colorbar(label='Wind Capacity (GW)')
    plt.title('Wind Power Capacity Potential (Downsampled)')
    plt.xlabel('Longitude (index)')
    plt.ylabel('Latitude (index)')
    plt.show()
    plt.savefig('wind_capacity_potential_downsampled.png', dpi=300)

# 光伏

In [ ]:
def pv_power(T, S, wind10):  # 光伏小时容量因子计算函数
    # T: 环境温度 (摄氏度)
    # S: 太阳辐射量 (单位：W/m²)
    # wind10: 10米高度的风速 (单位：m/s)
    
    # 计算光伏组件温度 (T_p)
    # T_p 表示组件的实际温度，它受环境温度、太阳辐射量以及风速的影响
    T_p = 4.3 + 0.943 * T + 0.028 * S / 3600 - 1.528 * wind10

    # 计算温度对光伏组件效率的影响 (TEM)
    # 组件温度高于25度会降低效率，每升高1度，效率降低0.5%
    TEM = (1 - 0.005 * (T_p - 25))

    # 系统效率 (SYS)
    # 包括逆变器、线缆等系统损耗，假设为85%
    SYS = 0.85

    # 返回光伏小时容量因子
    return TEM * SYS


def pv_power1(T, S, wind10):  # 光伏小时容量因子计算函数（改进版）
    # T: 环境温度 (摄氏度)
    # S: 太阳辐射量 (单位：W/m²)
    # wind100: 10米高度的风速 (单位：m/s)

    # 计算光伏组件温度 (T_p)
    # 与前一个函数相同，计算组件温度
    T_p = 4.3 + 0.943 * T + 0.028 * S / 3600 - 1.528 * wind10

    # 计算温度对光伏组件效率的影响 (TEM)
    TEM = (1 - 0.005 * (T_p - 25))

    # 系统效率 (SYS)
    SYS = 0.85

    # 返回光伏小时容量因子
    # 在这里，计算的单位从 W 转换为 kWh，S/1000 将 S 单位从 W/m² 转换为 kW/m²，除以 3600 以获得小时数的因子
    return S / 1000 / 3600 * TEM * SYS



In [ ]:
from astropy.coordinates import EarthLocation, AltAz, get_sun
from astropy.time import Time, TimeDelta
from astropy import units as u
import numpy as np
from numpy import radians

def PF(lat, lon, height):
    sigma_N = 1.3793 + lat* (1.2011 + lat * (-0.014404 + lat * 0.000080509))
    sigma_S = -0.41657 + lat* (1.41216 + lat * (0.024051 + lat * 0.00021828))
    # 创建观测点对象
    location = EarthLocation(lat=lat * u.deg, lon=lon * u.deg, height=0 * u.m)

    # 计算该经度对应的 UTC 偏移时间（小时）
    utc_offset_hours = lon / 15.0
    # 计算当地时间下午3点对应的 UTC 时间
    local_time = Time("2020-9-23 15:00:00") - TimeDelta(utc_offset_hours * 3600, format='sec')

    # 获取太阳在ICRS坐标系中的位置
    sun_icrs = get_sun(local_time)

    # 将太阳的位置转换到当前观测点的地平坐标系（AltAz）
    sun_altaz = sun_icrs.transform_to(AltAz(location=location, obstime=local_time))

    # 提取高度角
    altitude = sun_altaz.alt.deg  # 转为度数 
    azimuth = sun_altaz.az.deg - 180

    #print(f"方位角: {azimuth}")  
    #print(f"高度角: {altitude}")
    
    # 计算光伏倾角调整因子 (Performance Factor, PF)
    # 公式中包含最优倾角 (sigma)、太阳高度角 (sun_altitude)、太阳方位角 (sun_azimuth)
    # 使用三角函数和角度转换来进行计算
    if lat>= 0:
        PF = 1 / (
            np.cos(radians(sigma_N)) +
            np.sin(radians(sigma_N)) / (np.tan(radians(altitude))) * np.cos(radians(azimuth))
        )
    else:
                PF = 1 / (
            np.cos(radians(sigma_S)) +
            np.sin(radians(sigma_S)) / (np.tan(radians(altitude))) * np.cos(radians(azimuth))
        )

    # 返回计算的调整因子 PF
    return PF



In [ ]:
# ============================================================
# 测试不同经纬度的PF值（光伏倾角调整因子）
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import EarthLocation, AltAz, get_sun
from astropy.time import Time, TimeDelta
from astropy import units as u
from math import radians, sin, cos, tan

# 使用前面定义的 PF_scalar 函数
# 如果还没运行过PF_scalar定义，这里重新定义一次
def PF_scalar_test(lat_scalar, lon_scalar):
    """处理单个点的 PF 计算（标量版本）"""
    # 计算北半球与南半球最优倾角
    sigma_N = 1.3793 + lat_scalar * (1.2011 + lat_scalar * (-0.014404 + lat_scalar * 0.000080509))
    sigma_S = -0.41657 + lat_scalar * (1.41216 + lat_scalar * (0.024051 + lat_scalar * 0.00021828))
    
    # 创建观测点
    location = EarthLocation(lat=lat_scalar * u.deg, lon=lon_scalar * u.deg, height=0 * u.m)

    # 经度换算为UTC偏移（小时）
    utc_offset_hours = lon_scalar / 15.0
    # 当地时间下午3点对应的UTC时间
    local_time = Time("2020-09-23 15:00:00") - TimeDelta(utc_offset_hours * 3600, format='sec')

    # 获取太阳位置并转换到当地地平坐标
    sun_icrs = get_sun(local_time)
    sun_altaz = sun_icrs.transform_to(AltAz(location=location, obstime=local_time))
    altitude = sun_altaz.alt.deg
    azimuth = (sun_altaz.az.deg - 180) % 360

    # 计算PF（使用标量条件判断）
    try:
        if lat_scalar >= 0:
            PF_value = 1 / (
                cos(radians(sigma_N)) +
                sin(radians(sigma_N)) / tan(radians(altitude)) * cos(radians(azimuth))
            )
        else:
            PF_value = 1 / (
                cos(radians(sigma_S)) +
                sin(radians(sigma_S)) / tan(radians(altitude)) * cos(radians(azimuth))
            )
        return float(PF_value)
    except:
        return np.nan

# 1. 测试特定城市的PF值
print("=" * 70)
print("测试世界主要城市的PF值（光伏倾角调整因子）")
print("=" * 70)

cities = [
    ("北京", 39.9, 116.4),
    ("上海", 31.2, 121.5),
    ("广州", 23.1, 113.3),
    ("乌鲁木齐", 43.8, 87.6),
    ("拉萨", 29.7, 91.1),
    ("东京", 35.7, 139.7),
    ("纽约", 40.7, -74.0),
    ("伦敦", 51.5, -0.1),
    ("悉尼", -33.9, 151.2),
    ("开罗", 30.0, 31.2),
    ("赤道（厄瓜多尔）", 0.0, -78.5),
    ("北极圈（挪威）", 69.6, 18.9),
]

results = []
for city_name, lat, lon in cities:
    pf = PF_scalar_test(lat, lon)
    results.append({"城市": city_name, "纬度": lat, "经度": lon, "PF": pf})
    print(f"{city_name:20s} | 纬度: {lat:7.2f}° | 经度: {lon:7.2f}° | PF: {pf:.4f}")

df_cities = pd.DataFrame(results)
print("=" * 70)

# 2. 创建全球PF分布网格图
print("\n正在计算全球PF分布网格...")
lats = np.linspace(-60, 60, 61)  # 从-60°到60°，每2度一个点
lons = np.linspace(-180, 180, 91)  # 从-180°到180°，每4度一个点

PF_grid = np.zeros((len(lats), len(lons)))
for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):
        PF_grid[i, j] = PF_scalar_test(lat, lon)
    if i % 10 == 0:
        print(f"  进度: {i}/{len(lats)} 纬度带计算完成")

print("计算完成！\n")

# 3. 可视化全球PF分布
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 3.1 全球PF热力图
ax1 = axes[0, 0]
im1 = ax1.contourf(lons, lats, PF_grid, levels=20, cmap='RdYlGn')
ax1.set_xlabel('经度 (°)')
ax1.set_ylabel('纬度 (°)')
ax1.set_title('全球光伏倾角调整因子 (PF) 分布')
ax1.grid(True, alpha=0.3)
plt.colorbar(im1, ax=ax1, label='PF值')

# 3.2 不同纬度的PF变化（固定经度0°）
ax2 = axes[0, 1]
pf_by_lat = [PF_scalar_test(lat, 0) for lat in lats]
ax2.plot(lats, pf_by_lat, 'b-', linewidth=2)
ax2.set_xlabel('纬度 (°)')
ax2.set_ylabel('PF值')
ax2.set_title('不同纬度的PF变化（经度=0°）')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='PF=1')
ax2.legend()

# 3.3 不同经度的PF变化（固定纬度30°N）
ax3 = axes[1, 0]
pf_by_lon_30n = [PF_scalar_test(30, lon) for lon in lons]
ax3.plot(lons, pf_by_lon_30n, 'g-', linewidth=2)
ax3.set_xlabel('经度 (°)')
ax3.set_ylabel('PF值')
ax3.set_title('不同经度的PF变化（纬度=30°N）')
ax3.grid(True, alpha=0.3)
ax3.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='PF=1')
ax3.legend()

# 3.4 城市PF值柱状图
ax4 = axes[1, 1]
colors = ['red' if lat >= 0 else 'blue' for lat in df_cities['纬度']]
bars = ax4.barh(range(len(df_cities)), df_cities['PF'], color=colors, alpha=0.7)
ax4.set_yticks(range(len(df_cities)))
ax4.set_yticklabels(df_cities['城市'], fontsize=9)
ax4.set_xlabel('PF值')
ax4.set_title('主要城市PF值对比（红=北半球，蓝=南半球）')
ax4.axvline(x=1.0, color='black', linestyle='--', alpha=0.5, label='PF=1')
ax4.grid(True, alpha=0.3, axis='x')
ax4.legend()

plt.tight_layout()
plt.savefig('PF_distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# 4. 统计分析
print("\n" + "=" * 70)
print("PF值统计分析")
print("=" * 70)
valid_pf = PF_grid[~np.isnan(PF_grid)]
print(f"全局PF统计：")
print(f"  最小值: {np.nanmin(PF_grid):.4f}")
print(f"  最大值: {np.nanmax(PF_grid):.4f}")
print(f"  平均值: {np.nanmean(PF_grid):.4f}")
print(f"  中位数: {np.nanmedian(PF_grid):.4f}")
print(f"  标准差: {np.nanstd(PF_grid):.4f}")
print("=" * 70)

In [ ]:
# ============================================================
# 1 km 光伏潜力计算（PV）：0.25° -> 1 km 插值 + 约束 + 面积精算（PF预计算优化版）
# ============================================================
import os
import gc
import numpy as np
import xarray as xr
from pathlib import Path
from math import radians, sin, cos, tan
from astropy.time import Time, TimeDelta
from astropy.coordinates import EarthLocation, AltAz, get_sun
import astropy.units as u
from scipy.interpolate import RegularGridInterpolator

# ------------------[ 基本参数（与原版一致） ]------------------
PV_Rated_capacity = 265.0   # W/块
PV_S = 1.64                 # m^2/块
PR_DEFAULT = 0.82

DATA_DIR = Path(r"D:\Workspace")
FILE_PATTERN = "average_2023_segment_*.nc"
SSRD_VAR_NAME = "ssrd_adjust_avg"
TIME_SLICE = slice(0, 8760)
VERBOSE = True

# ------------------[ dask/xarray 选项与分块 ]------------------
# 1 km 建议块：512x512（按内存调整）
CHUNK_1KM = {"latitude": 512, "longitude": 512}
# 0.25° 建议块：经纬 200 左右，时间合并成一块（因为我们先sum再interp）
CHUNK_025D = {"time": -1, "latitude": 200, "longitude": 200}
xr.set_options(keep_attrs=True)

# ------------------[ 工具函数 ]------------------
def standardize_latlon_names(obj):
    mapping = {}
    if hasattr(obj, "dims"):
        if "lat" in obj.dims: mapping["lat"] = "latitude"
        if "lon" in obj.dims: mapping["lon"] = "longitude"
        if "y"   in obj.dims: mapping["y"]   = "latitude"
        if "x"   in obj.dims: mapping["x"]   = "longitude"
    if hasattr(obj, "coords"):
        if "lat" in obj.coords: mapping["lat"] = "latitude"
        if "lon" in obj.coords: mapping["lon"] = "longitude"
        if "y"   in obj.coords: mapping["y"]   = "latitude"
        if "x"   in obj.coords: mapping["x"]   = "longitude"
    try:
        obj = obj.rename(mapping)
    except Exception:
        pass
    return obj

def align_longitudes(obj, ref):
    ref_lon = ref["longitude"]
    to_0360 = (float(ref_lon.min()) >= 0) and (float(ref_lon.max()) <= 360)
    def _wrap(lon_vals, zero_to_360):
        lon = (lon_vals % 360 + 360) % 360
        if zero_to_360:
            return lon
        else:
            return xr.where(lon > 180, lon - 360, lon)
    new_lon = _wrap(obj["longitude"], to_0360)
    sorter = np.argsort(new_lon.values)
    return obj.assign_coords(longitude=new_lon).isel(longitude=sorter)

def cell_area_km2_row(lat_1d: xr.DataArray) -> xr.DataArray:
    """返回纬向条带面积（单位：km² / rad_lon），仅随纬度变。"""
    R = 6371.0
    dlat = float(np.abs(np.diff(lat_1d.values)).mean())
    dlat_rad = np.deg2rad(dlat)
    lat_ctr = np.deg2rad(lat_1d.values)
    lat_hi  = lat_ctr + 0.5 * dlat_rad
    lat_lo  = lat_ctr - 0.5 * dlat_rad
    strip = (R**2) * (np.sin(lat_hi) - np.sin(lat_lo))  # km² per rad(lon)
    return xr.DataArray(strip, dims=["latitude"], coords={"latitude": lat_1d}, name="area_strip_km2")

def get_time_dim(da: xr.DataArray) -> str:
    for cand in ("valid_time", "time", "hour", "t"):
        if cand in da.dims:
            return cand
    raise KeyError(f"未找到时间维；现有维度：{da.dims}")

def build_template_from_numpy(arr_like, *, lat_1d=None, lon_1d=None,
                              resolution_deg=1/120, bounds=None, name="template"):
    arr = np.asarray(arr_like) if not isinstance(arr_like, xr.DataArray) else arr_like.values
    nlat, nlon = arr.shape
    if lat_1d is not None and lon_1d is not None:
        lat = np.asarray(lat_1d); lon = np.asarray(lon_1d)
    else:
        if bounds is None:
            min_lat, max_lat = -90.0, 90.0
            min_lon, max_lon = -180.0, 180.0
        else:
            min_lat, max_lat, min_lon, max_lon = bounds
        lat = np.linspace(min_lat + resolution_deg/2, max_lat - resolution_deg/2, nlat)
        lon = np.linspace(min_lon + resolution_deg/2, max_lon - resolution_deg/2, nlon)
    da = xr.DataArray(
        np.zeros((nlat, nlon), dtype=np.float32),
        dims=["latitude", "longitude"],
        coords={"latitude": lat, "longitude": lon},
        name=name
    )
    return da.chunk(CHUNK_1KM)

def to_da_like(var, tmpl: xr.DataArray, name: str) -> xr.DataArray:
    if isinstance(var, xr.DataArray):
        var = standardize_latlon_names(var)
        try:
            var = var.reindex_like(tmpl, copy=False)
        except Exception:
            pass
        return var.astype("float32").chunk(CHUNK_1KM).rename(name)
    else:
        return xr.DataArray(var, dims=tmpl.dims, coords=tmpl.coords, name=name).astype("float32").chunk(CHUNK_1KM)

# ------------------[ PF 标量计算函数（仅用于预计算） ]------------------
def PF_scalar(lat_scalar, lon_scalar):
    """处理单个点的 PF 计算（标量版本）"""
    try:
        # 计算北半球与南半球最优倾角
        sigma_N = 1.3793 + lat_scalar * (1.2011 + lat_scalar * (-0.014404 + lat_scalar * 0.000080509))
        sigma_S = -0.41657 + lat_scalar * (1.41216 + lat_scalar * (0.024051 + lat_scalar * 0.00021828))
        
        # 创建观测点
        location = EarthLocation(lat=lat_scalar * u.deg, lon=lon_scalar * u.deg, height=0 * u.m)

        # 经度换算为UTC偏移（小时）
        utc_offset_hours = lon_scalar / 15.0
        # 当地时间下午3点对应的UTC时间
        local_time = Time("2020-09-23 15:00:00") - TimeDelta(utc_offset_hours * 3600, format='sec')

        # 获取太阳位置并转换到当地地平坐标
        sun_icrs = get_sun(local_time)
        sun_altaz = sun_icrs.transform_to(AltAz(location=location, obstime=local_time))
        altitude = sun_altaz.alt.deg
        azimuth = (sun_altaz.az.deg - 180) % 360  # 标准化方位角

        # 计算PF（使用标量条件判断）
        if lat_scalar >= 0:
            PF_value = 1 / (
                cos(radians(sigma_N)) +
                sin(radians(sigma_N)) / tan(radians(altitude)) * cos(radians(azimuth))
            )
        else:
            PF_value = 1 / (
                cos(radians(sigma_S)) +
                sin(radians(sigma_S)) / tan(radians(altitude)) * cos(radians(azimuth))
            )

        return float(PF_value)
    except:
        return 1.0  # 异常时返回默认值

# ------------------[ 步骤 1：预计算全球 PF 网格 ]------------------
print("=" * 70)
print("[PV] 步骤 1/3：预计算全球 PF 网格（约需 1-2 分钟）...")
print("=" * 70)

# 创建稀疏网格（每 0.5° 一个点，约 360x720 = 26万点）
lat_sparse = np.arange(-90, 90.5, 0.5)
lon_sparse = np.arange(-180, 180.5, 0.5)

print(f"网格大小: {len(lat_sparse)} x {len(lon_sparse)} = {len(lat_sparse)*len(lon_sparse):,} 个点")
print("开始计算...")

PF_sparse = np.zeros((len(lat_sparse), len(lon_sparse)), dtype=np.float32)

for i, lat in enumerate(lat_sparse):
    for j, lon in enumerate(lon_sparse):
        PF_sparse[i, j] = PF_scalar(lat, lon)
    
    # 每计算 18° 打印一次进度（每36行）
    if i % 36 == 0:
        progress = 100 * i / len(lat_sparse)
        print(f"  进度: {i}/{len(lat_sparse)} ({progress:.0f}%) - 当前纬度: {lat:.1f}°")

print(f"  进度: {len(lat_sparse)}/{len(lat_sparse)} (100%) - 完成！")

# 创建插值器
pf_interpolator = RegularGridInterpolator(
    (lat_sparse, lon_sparse), 
    PF_sparse, 
    method='linear', 
    bounds_error=False, 
    fill_value=1.0
)

# 定义快速 PF 函数（使用插值）
def PF_fast(lat_arr, lon_arr):
    """通过插值快速获取 PF 值"""
    points = np.column_stack([lat_arr.ravel(), lon_arr.ravel()])
    pf_vals = pf_interpolator(points).reshape(lat_arr.shape)
    return pf_vals.astype(np.float32)

print("\n✓ PF 网格预计算完成！")
print(f"  PF 值范围: [{np.nanmin(PF_sparse):.4f}, {np.nanmax(PF_sparse):.4f}]")
print(f"  PF 平均值: {np.nanmean(PF_sparse):.4f}\n")

# ------------------[ 步骤 2：准备 1 km 输入模板 ]------------------
print("=" * 70)
print("[PV] 步骤 2/3：准备 1 km 输入数据...")
print("=" * 70)

lucc_solar_src = globals()["lucc_solar"]
if isinstance(lucc_solar_src, xr.DataArray) and "latitude" in lucc_solar_src.coords and "longitude" in lucc_solar_src.coords:
    template = build_template_from_numpy(
        lucc_solar_src,
        lat_1d=lucc_solar_src.latitude.values,
        lon_1d=lucc_solar_src.longitude.values,
        resolution_deg=1/120,
    )
else:
    template = build_template_from_numpy(lucc_solar_src, resolution_deg=1/120)

lucc_da        = to_da_like(globals()["lucc_solar"], template, "lucc_solar")
solar_slope_da = to_da_like(globals()["solar_slope"], template, "solar_slope")
elevation_da   = to_da_like(globals()["elevation"], template, "elevation")
protected_da   = to_da_like(globals()["protected"], template, "protected")

lat_1km_all = template["latitude"]
lon_1km_all = template["longitude"]

print(f"模板形状: {template.shape}")
print(f"纬度范围: [{float(lat_1km_all.min()):.3f}, {float(lat_1km_all.max()):.3f}]")
print(f"经度范围: [{float(lon_1km_all.min()):.3f}, {float(lon_1km_all.max()):.3f}]")

# 预先计算纬向条带面积（km²/rad_lon）
area_strip_km2 = cell_area_km2_row(lat_1km_all).astype("float32").chunk({"latitude": CHUNK_1KM["latitude"]})

print("\n✓ 输入数据准备完成！\n")

# ------------------[ 步骤 3：读取 0.25° 分段并计算 ]------------------
print("=" * 70)
print("[PV] 步骤 3/3：处理 0.25° 分段文件...")
print("=" * 70)

files = sorted(str(p) for p in DATA_DIR.glob(FILE_PATTERN))
print(f"匹配到 {len(files)} 个文件\n")

if not files:
    raise FileNotFoundError(f"未匹配到文件：{DATA_DIR}\\{FILE_PATTERN}")

GE_pv_tiles = []
CP_pv_tiles = []
ok = 0
skip = 0

for seg_idx, seg_path in enumerate(files):
    seg_name = Path(seg_path).name
    
    try:
        # 读取数据
        ds0 = xr.open_dataset(seg_path, engine="netcdf4", chunks=CHUNK_025D)
        ds0 = standardize_latlon_names(ds0)

        if SSRD_VAR_NAME not in ds0:
            skip += 1
            if VERBOSE:
                print(f"[{seg_idx+1}/{len(files)}] SKIP - {seg_name}: 无变量 '{SSRD_VAR_NAME}'")
            continue

        ds = align_longitudes(ds0, template)
        SSRD = ds[SSRD_VAR_NAME].astype("float32")

        tdim = get_time_dim(SSRD)
        SSRD_sum_025d = SSRD.isel({tdim: TIME_SLICE}).sum(dim=tdim)

        # 目标 1 km 网格
        eps = 1e-9
        lon_target = lon_1km_all.sel(longitude=slice(
            float(ds.longitude.min()) - eps,
            float(ds.longitude.max()) + eps
        ))
        lat_target = lat_1km_all

        # 插值与裁剪
        dlon_deg = float(np.abs(np.diff(lon_target.values)).mean())
        dlon_rad = np.deg2rad(dlon_deg)
        
        SSRD_sum = SSRD_sum_025d.interp(
            latitude=lat_target, 
            longitude=lon_target, 
            method="linear"
        ).chunk(CHUNK_1KM)

        lucc_chunk        = lucc_da.sel(latitude=lat_target, longitude=lon_target)
        solar_slope_chunk = solar_slope_da.sel(latitude=lat_target, longitude=lon_target)
        elevation_chunk   = elevation_da.sel(latitude=lat_target, longitude=lon_target)
        protected_chunk   = protected_da.sel(latitude=lat_target, longitude=lon_target)

        # 面积计算
        cell_area_km2 = (area_strip_km2 * dlon_rad).broadcast_like(lucc_chunk)
        cell_area_m2 = (cell_area_km2 * 1e6).astype("float32")

        # ===== 使用预计算的 PF 网格（插值） =====
        lat_2d, lon_2d = np.meshgrid(
            lat_target.values,
            lon_target.values,
            indexing='ij'
        )
        PF_pv_values = PF_fast(lat_2d, lon_2d)
        PF_pv = xr.DataArray(
            PF_pv_values,
            dims=["latitude", "longitude"],
            coords={"latitude": lat_target, "longitude": lon_target},
            name="PF_pv"
        ).astype("float32")

        # 可建设掩膜
        buildable = (lucc_chunk * solar_slope_chunk * elevation_chunk * protected_chunk).astype("float32")

        # 计算发电量与装机
        GE_pv = (cell_area_m2 * buildable * PF_pv * SSRD_sum * (0.1 / 3.6 / 1e18)).rename("GE_pv").chunk(CHUNK_1KM)
        CP_pv = (cell_area_m2 / PV_S * PV_Rated_capacity * buildable * PF_pv / 1e9).rename("CP_pv").chunk(CHUNK_1KM)

        GE_pv_tiles.append(GE_pv)
        CP_pv_tiles.append(CP_pv)
        ok += 1

        if VERBOSE:
            print(f"[{seg_idx+1}/{len(files)}] ✓ {seg_name} | shape={GE_pv.shape} | "
                  f"lon=[{float(lon_target.min()):.1f}, {float(lon_target.max()):.1f}]")

        # 清理内存
        del ds0, ds, SSRD, SSRD_sum_025d, SSRD_sum
        del lucc_chunk, solar_slope_chunk, elevation_chunk, protected_chunk
        del cell_area_km2, cell_area_m2, lat_2d, lon_2d, PF_pv_values, PF_pv, buildable
        gc.collect()

    except Exception as e:
        skip += 1
        print(f"[{seg_idx+1}/{len(files)}] ✗ ERROR - {seg_name}: {e}")

print("\n" + "=" * 70)
print(f"处理完成：成功 {ok} 段，跳过 {skip} 段")
print("=" * 70 + "\n")

# ------------------[ 拼接与保存 ]------------------
if not GE_pv_tiles:
    raise ValueError(
        "没有任何分段成功处理，无法拼接。\n"
        f"请检查：路径/通配符，变量名 '{SSRD_VAR_NAME}'，时间维，或经度系。"
    )

print("正在拼接所有 tile...")
GE_pv_all = xr.concat(GE_pv_tiles, dim="longitude", coords="minimal",
                      compat="override", join="override")
CP_pv_all = xr.concat(CP_pv_tiles, dim="longitude", coords="minimal",
                      compat="override", join="override")

# De-dup longitude seams from segmented concat
GE_pv_all = GE_pv_all.groupby("longitude").mean()
CP_pv_all = CP_pv_all.groupby("longitude").mean()


if VERBOSE:
    print(f"拼接完成：{dict(GE_pv_all.sizes)}\n")

# 翻转纬度符号并升序排列
def flip_lat_sign_and_sort(da: xr.DataArray) -> xr.DataArray:
    if "latitude" not in da.coords:
        return da
    da = da.assign_coords(latitude=-1.0 * da["latitude"])
    return da.sortby(['latitude', 'longitude'])

print("正在翻转纬度并排序...")
GE_pv_all = flip_lat_sign_and_sort(GE_pv_all).chunk(CHUNK_1KM)
CP_pv_all = flip_lat_sign_and_sort(CP_pv_all).chunk(CHUNK_1KM)

# 写盘（开启压缩）
print("\n正在保存结果到磁盘（这可能需要几分钟）...")
enc_GE = {"GE_pv": {"dtype": "float32", "zlib": True, "complevel": 4,
                    "chunksizes": (CHUNK_1KM["latitude"], CHUNK_1KM["longitude"])}}
enc_CP = {"CP_pv": {"dtype": "float32", "zlib": True, "complevel": 4,
                    "chunksizes": (CHUNK_1KM["latitude"], CHUNK_1KM["longitude"])}}

ge_path = DATA_DIR / "GE_pv_1km_PWh.nc"
cp_path = DATA_DIR / "CP_pv_1km_GW.nc"

GE_pv_all.to_netcdf(ge_path, encoding=enc_GE)
CP_pv_all.to_netcdf(cp_path, encoding=enc_CP)

print("\n" + "=" * 70)
print("✓ 光伏潜力计算完成！")
print("=" * 70)
print(f"发电量文件: {ge_path}")
print(f"装机文件: {cp_path}")
print("=" * 70)

In [ ]:
# Downsample the data for visualization and replace NaN with 0
downsample_factor = 10  # Adjust this value based on your needs
CP_pv_downsampled = CP_pv_all[::downsample_factor, ::downsample_factor]

# Replace NaN values with 0
#CP_pv_downsampled = CP_pv_downsampled.fillna(0)

# 排序坐标以确保按递增顺序
#CP_pv_downsampled = CP_pv_downsampled.sortby(['latitude', 'longitude'])

# 先检查数据的基本信息
print(f"CP_pv_downsampled shape: {CP_pv_downsampled.shape}")
print(f"Data type: {CP_pv_downsampled.dtype}")
print(f"Min value: {CP_pv_downsampled.min().values}")
print(f"Max value: {CP_pv_downsampled.max().values}")
print(f"Has NaN values: {CP_pv_downsampled.isnull().any().values}")
print(f"Non-zero values count: {(CP_pv_downsampled > 0).sum().values}")

# 检查坐标顺序
print(f"Latitude range: {CP_pv_downsampled.latitude.min().values} to {CP_pv_downsampled.latitude.max().values}")
print(f"Longitude range: {CP_pv_downsampled.longitude.min().values} to {CP_pv_downsampled.longitude.max().values}")

plt.figure(figsize=(15, 10))
try:
    CP_pv_downsampled.plot(cmap='viridis', robust=True)
    plt.title('PV Power Capacity Potential (Downsampled)')
    plt.show()
except Exception as e:
    print(f"xarray plot failed: {e}")
    # 备用方案：使用matplotlib直接绘图
    plt.imshow(CP_pv_downsampled.values, cmap='viridis', aspect='auto', origin='lower')
    plt.colorbar(label='PV Capacity (GW)')
    plt.title('PV Power Capacity Potential (Downsampled)')
    plt.xlabel('Longitude (index)')
    plt.ylabel('Latitude (index)')
    plt.show()
    plt.savefig('pv_capacity_potential_downsampled.png', dpi=300)

In [ ]:
import xarray as xr

# 读取风电装机容量
cp_wind = xr.open_dataarray(r'd:\Workspace\CP_wind_1km_GW.nc')
print(cp_wind)

# 读取光伏发电量
cp_pv = xr.open_dataarray(r'd:\Workspace\CP_pv_1km_GW.nc')
print(cp_pv)

In [ ]:
import matplotlib.pyplot as plt

# Downsample for visualization
cp_wind_down = cp_wind[::10, ::10].fillna(0)
cp_pv_down = cp_pv[::10, ::10].fillna(0)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
cp_wind_down.plot(cmap='viridis', robust=True)
plt.title('Wind Capacity Potential (GW, downsampled)')

plt.subplot(1, 2, 2)
cp_pv_down.plot(cmap='plasma', robust=True)
plt.title('PV Capacity Potential (GW, downsampled)')

plt.tight_layout()
plt.show()

In [ ]:
# 全局求和并比较风电与光伏装机潜力
wind_sum = float(cp_wind.sum().values)
pv_sum = float(cp_pv.sum().values)
print(f"风电装机潜力总和: {wind_sum/1000:.2e} TW")
print(f"光伏装机潜力总和: {pv_sum/1000:.2e} TW")
print(f"风电/光伏装机潜力比值: {wind_sum/pv_sum:.2f}")

In [ ]:
import numpy as np
import xarray as xr

# 假设 CP_wind_1km_GW.nc 已经是 (latitude, longitude) 坐标，单位为 GW
cp_wind_1km = xr.open_dataarray("D:/Workspace/CP_wind_1km_GW.nc")
cp_pv_1km = xr.open_dataarray("D:/Workspace/CP_pv_1km_GW.nc")

# --- 调试信息：打印原始数据信息 ---
print("--- Original Data Info ---")
print("Wind dims:", cp_wind_1km.dims)
print("Wind coords:", cp_wind_1km.coords)
# --------------------------------

# --- 修复坐标名称和范围 ---
def fix_coords(da):
    # 1. 处理维度名称 (y -> latitude, x -> longitude)
    rename_dims = {}
    if 'y' in da.dims and 'latitude' not in da.dims: rename_dims['y'] = 'latitude'
    if 'x' in da.dims and 'longitude' not in da.dims: rename_dims['x'] = 'longitude'
    if 'lat' in da.dims and 'latitude' not in da.dims: rename_dims['lat'] = 'latitude'
    if 'lon' in da.dims and 'longitude' not in da.dims: rename_dims['lon'] = 'longitude'
    
    if rename_dims:
        print(f"Renaming dims: {rename_dims}")
        da = da.rename(rename_dims)

    # 2. 处理坐标名称 (跟维度名一致)
    rename_coords = {}
    if 'y' in da.coords and 'latitude' not in da.coords: rename_coords['y'] = 'latitude'
    if 'x' in da.coords and 'longitude' not in da.coords: rename_coords['x'] = 'longitude'
    if 'lat' in da.coords and 'latitude' not in da.coords: rename_coords['lat'] = 'latitude'
    if 'lon' in da.coords and 'longitude' not in da.coords: rename_coords['lon'] = 'longitude'
    
    if rename_coords:
        print(f"Renaming coords: {rename_coords}")
        da = da.rename(rename_coords)
    
    # 3. 检查并修复缺失的 latitude 坐标
    if 'latitude' in da.dims and 'latitude' not in da.coords:
        print("Warning: 'latitude' dimension exists but coordinate is missing. Attempting to reconstruct...")
        ny = da.sizes['latitude']
        # 尝试从 longitude 推断分辨率
        if 'longitude' in da.coords:
            lons = da.longitude.values
            res = abs(lons[1] - lons[0])
            print(f"Inferred resolution from longitude: {res}")
            # 假设是全球覆盖，或者根据分辨率推算
            # 如果是 21600 行，分辨率 ~0.00833 (30 arcsec)，覆盖 180 度
            # 通常是从 90 到 -90 (Top-down)
            # 简单的构建：
            start_lat = 90 - res/2
            lats = np.linspace(start_lat, start_lat - (ny-1)*res, ny)
            
            # 检查范围是否合理
            if lats.min() < -90.1 or lats.max() > 90.1:
                 print("Warning: Reconstructed latitude out of range [-90, 90]. Check assumptions.")
            
            da = da.assign_coords(latitude=lats)
            print(f"Assigned latitude coords: {da.latitude.values[0]} to {da.latitude.values[-1]}")
        else:
            print("Error: Cannot reconstruct latitude because longitude is also missing.")

    # 4. 处理经度 0-360 转 -180-180
    if 'longitude' in da.coords:
        lon_max = da.longitude.max().values
        if lon_max > 180.1: # 稍微放宽一点容差
            print("Detected 0-360 longitude, converting to -180-180...")
            da = da.assign_coords(longitude=(((da.longitude + 180) % 360) - 180))
            da = da.sortby('longitude')
            
    return da

cp_wind_1km = fix_coords(cp_wind_1km)
cp_pv_1km = fix_coords(cp_pv_1km)
# ------------------------

# 构造新的 0.25° 网格
lat_bins = np.arange(-90, 90.25, 0.25)
lon_bins = np.arange(-180, 180.25, 0.25)

# 用 groupby_bins 聚合
print("Aggregating Wind data...")
if 'latitude' not in cp_wind_1km.coords:
    print("Error: 'latitude' coordinate still missing for Wind data. Aggregation will fail.")
else:
    cp_wind_lat = cp_wind_1km.groupby_bins(
        "latitude", lat_bins, labels=lat_bins[:-1], right=False
    ).sum()

    cp_wind_025d = cp_wind_lat.groupby_bins(
        "longitude", lon_bins, labels=lon_bins[:-1], right=False
    ).sum()
    
    # 检查中间结果
    print(f"Wind sum after aggregation: {cp_wind_025d.sum().values}")

print("Aggregating PV data...")
if 'latitude' not in cp_pv_1km.coords:
    print("Error: 'latitude' coordinate still missing for PV data. Aggregation will fail.")
else:
    cp_pv_lat = cp_pv_1km.groupby_bins(
        "latitude", lat_bins, labels=lat_bins[:-1], right=False
    ).sum()

    cp_pv_025d = cp_pv_lat.groupby_bins(
        "longitude", lon_bins, labels=lon_bins[:-1], right=False
    ).sum()
    
    print(f"PV sum after aggregation: {cp_pv_025d.sum().values}")

# 可选：重命名坐标为 latitude/longitude
if 'latitude_bins' in cp_wind_025d.coords:
    cp_wind_025d = cp_wind_025d.rename({"latitude_bins": "latitude", "longitude_bins": "longitude"})
if 'latitude_bins' in cp_pv_025d.coords:
    cp_pv_025d = cp_pv_025d.rename({"latitude_bins": "latitude", "longitude_bins": "longitude"})

# 保存为新文件
cp_wind_025d.to_netcdf("D:/Workspace/CP_wind_025d_GW.nc")
cp_pv_025d.to_netcdf("D:/Workspace/CP_pv_025d_GW.nc")
print("✓ 已聚合并保存为 0.25° 网格")

In [ ]:
# 0.25度分辨率聚合后绘图与求和验证
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# 假设 cp_wind_025d, cp_pv_025d 已经按0.25度聚合
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# 绘制风能分布
im1 = ax[0].imshow(cp_wind_025d.values, origin='lower', aspect='auto', cmap='viridis')
ax[0].set_title('风能0.25度分布')
plt.colorbar(im1, ax=ax[0])

# 绘制光伏分布
im2 = ax[1].imshow(cp_pv_025d.values, origin='lower', aspect='auto', cmap='plasma')
ax[1].set_title('光伏0.25度分布')
plt.colorbar(im2, ax=ax[1])

plt.tight_layout()

# 求和验证
wind_sum = np.nansum(cp_wind_025d.values)
pv_sum = np.nansum(cp_pv_025d.values)
print(f'风能总和: {wind_sum/1000}TW')
print(f'光伏总和: {pv_sum/1000}TW')

In [ ]:
import pandas as pd
import numpy as np

# 读取结果表格
input_path = 'DataOutput/Used_output/combined_RegionRE_LowAE_AddSteam_methanol_results.csv'
df = pd.read_csv(input_path)

# 假设 cp_wind_025d, cp_pv_025d 已经是 xarray.DataArray，坐标为0.25度网格中心，单位GW
lat_centers = cp_wind_025d['latitude'].values
lon_centers = cp_wind_025d['longitude'].values

# --- 2) 经度对齐函数：把 df 的经度对齐到 DA 的经度域 ---
def align_lon_to_da(lon_value, da_lon_vals):
    lon_min, lon_max = da_lon_vals.min(), da_lon_vals.max()
    if lon_min >= 0 and lon_max <= 360:
        # DA: 0..360，把 -180..180 的经度转到 0..360
        lon_aligned = lon_value % 360
        if lon_aligned < 0:  # 保险处理
            lon_aligned += 360
        return lon_aligned
    else:
        # DA: -180..180，把 0..360 的经度转到 -180..180
        lon_mod = lon_value % 360
        if lon_mod > 180:
            lon_mod -= 180*2
        return lon_mod
    
def get_grid_center(val, centers):
    """找到最近的网格中心"""
    return centers[np.abs(centers - val).argmin()]

def get_potential_interp(lat, lon, da):
    lon = align_lon_to_da(lon, da['longitude'].values)
    v = da.interp(latitude=lat, longitude=lon, method='nearest')
    return float(np.asarray(v.values)) * 1e6



# 匹配风光装机潜力（单位转为kW）
df['wind_potential'] = df.apply(lambda row: get_potential_interp(row['latitude'], row['longitude'], cp_wind_025d), axis=1)
df['pv_potential'] = df.apply(lambda row: get_potential_interp(row['latitude'], row['longitude'], cp_pv_025d), axis=1)

# 计算装机量与潜力的系数
df['wind_coef'] = df['wind_potential'] / df['wind_capacity_kW']
df['pv_coef'] = df['pv_potential']  / df['pv_capacity_kW']

# 取风电和光伏中最小的系数，乘以10吨，作为生产潜力
df['production_potential_ton'] = df[['wind_coef', 'pv_coef']].min(axis=1) *10 / 1000/1000

# 保存新表格
output_path = input_path.replace('.csv', '_withPotenial.csv')
df.to_csv(output_path, index=False)
print(f'已保存: {output_path}')

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# ==========================================================
# 读取结果表格
# ==========================================================
input_path = 'DataOutput/Used_output/combined_RegionRE_LowAE_AddSteam_methanol_results.csv'
df = pd.read_csv(input_path)

# ==========================================================
# 获取网格中心坐标（假设 cp_wind_025d 和 cp_pv_025d 的坐标一致）
# ==========================================================
lat_centers = cp_wind_025d['latitude'].values
lon_centers = cp_wind_025d['longitude'].values
mesh_lat, mesh_lon = np.meshgrid(lat_centers, lon_centers, indexing='ij')
grid_points = np.column_stack([mesh_lat.ravel(), mesh_lon.ravel()])

# 构建KDTree用于快速最近点查找
tree = cKDTree(grid_points)

# ==========================================================
# 待匹配点坐标
# ==========================================================
data_points = df[['latitude', 'longitude']].values

# 初始最近邻查询
_, idx = tree.query(data_points)

# ==========================================================
# 展平潜力数据并转单位（GW → kW）
# ==========================================================
wind_flat = cp_wind_025d.values.ravel() * 1e6
pv_flat   = cp_pv_025d.values.ravel() * 1e6

wind_potential = np.zeros(len(df))
pv_potential   = np.zeros(len(df))

# ==========================================================
# 定义一个函数：如果潜力为 0，搜索周边非零格点
# ==========================================================
def find_nonzero_potential(i, arr_flat, ktree, data_points, max_k=10):
    """
    从最近邻开始，逐步扩大搜索半径，返回第一个非零潜力值。
    max_k 表示最多搜索多少个邻居（例如10个）。
    """
    dists, inds = ktree.query(data_points[i], k=max_k)
    for j in range(len(inds)):
        val = arr_flat[inds[j]]
        if np.isfinite(val) and val > 0:
            return val
    # 若全为0，返回0
    return 0.0

# ==========================================================
# 遍历每个点进行潜力匹配（带邻域回退）
# ==========================================================
for i in range(len(df)):
    wind_val = wind_flat[idx[i]]
    pv_val   = pv_flat[idx[i]]

    if wind_val <= 0:
        wind_val = find_nonzero_potential(i, wind_flat, tree, data_points)
    if pv_val <= 0:
        pv_val   = find_nonzero_potential(i, pv_flat, tree, data_points)

    wind_potential[i] = wind_val
    pv_potential[i]   = pv_val

# ==========================================================
# 写入结果并计算系数与生产潜力
# ==========================================================
df['wind_potential'] = wind_potential
df['pv_potential'] = pv_potential

df['wind_coef'] = df['wind_potential'] / df['wind_capacity_kW']
df['pv_coef']   = df['pv_potential'] / df['pv_capacity_kW']

# 取风电和光伏中最小系数乘以10吨，转为百万吨
df['production_potential_Mt'] = df[['wind_coef', 'pv_coef']].min(axis=1) * 10 / 1e6

# ==========================================================
# 保存结果
# ==========================================================
output_path = input_path.replace('.csv', '_withPotential.csv')
df.to_csv(output_path, index=False)
print(f'✅ 已保存: {output_path}')


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False   # 正确显示负号

# # 再次绘制空间分布图，支持中文标题和标签
# plt.figure(figsize=(8, 6))
# plt.scatter(mesh_lon.ravel(), mesh_lat.ravel(), c=wind_potential_flat, cmap='viridis', s=10, label='风电潜力网格')
# plt.colorbar(label='风电潜力(kW)')
# plt.xlabel('经度')
# plt.ylabel('纬度')
# plt.title('风电装机潜力空间分布')
# plt.legend()
# plt.show()

# # 仅绘制orig_idx不为0的CSV数据点
# plt.figure(figsize=(8, 6))
# mask = df['orig_idx'] != 0
# plt.scatter(df.loc[mask, 'longitude'], df.loc[mask, 'latitude'], c='red', s=0.5, label='orig_idx≠0数据点')
# plt.xlabel('经度')
# plt.ylabel('纬度')
# plt.title('orig_idx≠0的CSV数据空间分布')
# plt.legend()
# plt.show()

# 叠加风电潜力网格与orig_idx≠0的数据点
plt.figure(figsize=(8, 6))
plt.scatter(mesh_lon.ravel(), mesh_lat.ravel(), c=wind_potential_flat, cmap='viridis', s=10, label='风电潜力网格')
plt.scatter(df.loc[mask, 'longitude'], df.loc[mask, 'latitude'], c='red', s=0.5, label='orig_idx≠0数据点')
plt.colorbar(label='风电潜力(kW)')
plt.xlabel('经度')
plt.ylabel('纬度')
plt.title('风电潜力与orig_idx≠0数据空间对应关系')
plt.legend()
plt.show()

# 批量计算

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import glob
import os
import re
import xarray as xr

# 读取 0.25 度分辨率的风电和光伏装机潜力文件
cp_wind_025d = xr.open_dataarray(r"D:\Workspace\CP_wind_025d_GW.nc")
cp_pv_025d = xr.open_dataarray(r"D:\Workspace\CP_pv_025d_GW.nc")

# 获取所有 _results.csv 文件
input_dir = 'DataOutput/Used_output/'
result_files = glob.glob(os.path.join(input_dir, '*_results.csv'))

print(f"找到 {len(result_files)} 个结果文件:")
for f in result_files:
    print(f"  - {os.path.basename(f)}")

# 准备 KDTree（只需构建一次）
lat_centers = cp_wind_025d['latitude'].values
lon_centers = cp_wind_025d['longitude'].values
mesh_lat, mesh_lon = np.meshgrid(lat_centers, lon_centers, indexing='ij')
grid_points = np.column_stack([mesh_lat.ravel(), mesh_lon.ravel()])
tree = cKDTree(grid_points)

# 提取扁平化的潜力数据
wind_potential_flat = cp_wind_025d.values.ravel() * 1e6  # GW转kW
pv_potential_flat = cp_pv_025d.values.ravel() * 1e6

# 批量处理所有文件
for input_path in result_files:
    print(f"\n处理文件: {os.path.basename(input_path)}")
    
    try:
        # 读取数据
        df = pd.read_csv(input_path)
        
        original_count = len(df)
        
        # ⭐ 第一步：筛掉平准化成本为0的点
        # 查找成本列（支持多种命名格式）
        cost_col = None
        cost_patterns = [
            r'levelized.*cost.*per.*(kg|ton)',  # levelized_xxx_cost_per_kg/ton
            r'.*cost.*per.*(kg|ton)',           # xxx_cost_per_kg/ton
        ]
        
        for pattern in cost_patterns:
            for col in df.columns:
                if re.search(pattern, col, re.IGNORECASE):
                    cost_col = col
                    break
            if cost_col:
                break
        
        if cost_col:
            print(f"  找到成本列: {cost_col}")
            
            # 转换为数值并筛选
            df[cost_col] = pd.to_numeric(df[cost_col], errors='coerce')
            cost_zero_count = (df[cost_col] <= 0).sum() | df[cost_col].isna().sum()
            
            # 只保留成本 > 0 的点
            df = df[df[cost_col] > 0].copy()
            
            if cost_zero_count > 0:
                print(f"  ⚠️  筛掉成本为0或无效的点: {cost_zero_count} ({cost_zero_count/original_count*100:.2f}%)")
                print(f"  剩余数据点: {len(df)}")
        else:
            print(f"  警告: 未找到成本列，跳过成本筛选")
        
        if len(df) == 0:
            print(f"  跳过: 筛选后无有效数据")
            continue
        
        # 灵活匹配列名：优先使用 wind_capacity_kW，否则使用 wind_capacity
        wind_col = 'wind_capacity_kW' if 'wind_capacity_kW' in df.columns else 'wind_capacity'
        pv_col = 'pv_capacity_kW' if 'pv_capacity_kW' in df.columns else 'pv_capacity'
        
        # 检查必要列是否存在
        if wind_col not in df.columns or pv_col not in df.columns:
            print(f"  跳过: 缺少必要的容量列")
            print(f"  现有列: {list(df.columns)}")
            continue
        
        print(f"  使用容量列: {wind_col}, {pv_col}")
        
        # KDTree 查询最近网格
        data_points = df[['latitude', 'longitude']].values
        _, idx = tree.query(data_points)
        
        # 匹配风光潜力
        df['wind_potential'] = wind_potential_flat[idx]
        df['pv_potential'] = pv_potential_flat[idx]
        
        # 获取容量数据
        wind_capacity_kW = df[wind_col]
        pv_capacity_kW = df[pv_col]
        
        # ⭐⭐⭐ 关键修复：只筛掉"一端是毛刺且另一端潜力为0"的点 ⭐⭐⭐
        # 设定最小有效容量阈值（1 kW）
        min_capacity_threshold = 1.0  # kW
        
        # 判断是否为毛刺容量
        wind_is_tiny = wind_capacity_kW < min_capacity_threshold
        pv_is_tiny = pv_capacity_kW < min_capacity_threshold
        
        # 判断潜力是否为0
        wind_potential_zero = df['wind_potential'] <= 0
        pv_potential_zero = df['pv_potential'] <= 0
        
        # ⭐ 核心逻辑：只在以下情况设置 coef = inf（表示该路径不可行）
        # 1. 容量是毛刺 AND 潜力为0  → 明确不可行
        # 2. 容量 >= 阈值但潜力为0    → 潜力限制
        # 3. 容量是毛刺 AND 潜力 > 0  → 保留！计算 coef
        
        # 风电系数：只有当"容量太小且潜力为0"或"潜力为0"时才设为 inf
        df['wind_coef'] = np.where(
            (wind_is_tiny & wind_potential_zero) | wind_potential_zero,
            np.inf,
            np.where(
                wind_is_tiny & (df['wind_potential'] > 0),
                df['wind_potential'] / min_capacity_threshold,  # 使用阈值避免除以极小值
                df['wind_potential'] / wind_capacity_kW
            )
        )
        
        # 光伏系数：同样逻辑
        df['pv_coef'] = np.where(
            (pv_is_tiny & pv_potential_zero) | pv_potential_zero,
            np.inf,
            np.where(
                pv_is_tiny & (df['pv_potential'] > 0),
                df['pv_potential'] / min_capacity_threshold,  # 使用阈值避免除以极小值
                df['pv_potential'] / pv_capacity_kW
            )
        )
        
        # 取最小系数（选择更限制的资源）
        min_coef = df[['wind_coef', 'pv_coef']].min(axis=1)
        
        # 设定合理的系数上限（避免异常大的值）
        max_reasonable_coef = 1e10
        
        # 计算生产潜力
        # ⚠️ 注意：所有产品（hydrogen, ammonia, methanol, FTL）的装机容量都是针对 10 吨（10,000 kg）的年产量优化的
        # min_coef 表示实际装机潜力与优化装机容量的比值
        # 因此，生产潜力 = min_coef × 10 吨
        df['production_potential_ton'] = np.where(
            np.isfinite(min_coef) & (min_coef < max_reasonable_coef),
            min_coef * 10,  # 乘以10吨（目标产量）
            0
        )
        
        # 统计信息
        total_points = len(df)
        valid_points = (df['production_potential_ton'] > 0).sum()
        filtered_points = total_points - valid_points
        
        # 详细统计各种情况
        wind_tiny_and_zero = (wind_is_tiny & wind_potential_zero).sum()
        pv_tiny_and_zero = (pv_is_tiny & pv_potential_zero).sum()
        wind_tiny_but_has_potential = (wind_is_tiny & (df['wind_potential'] > 0)).sum()
        pv_tiny_but_has_potential = (pv_is_tiny & (df['pv_potential'] > 0)).sum()
        both_inf = ((df['wind_coef'] == np.inf) & (df['pv_coef'] == np.inf)).sum()
        coef_too_large = ((min_coef >= max_reasonable_coef) & np.isfinite(min_coef)).sum()
        
        print(f"\n  潜力计算后统计:")
        print(f"    总数据点: {total_points}")
        print(f"    有效数据点: {valid_points} ({valid_points/total_points*100:.2f}%)")
        print(f"    过滤的点: {filtered_points} ({filtered_points/total_points*100:.2f}%)")
        
        print(f"\n  容量毛刺分析:")
        print(f"    ├─ 风电容量是毛刺且潜力为0 (已筛掉): {wind_tiny_and_zero}")
        print(f"    ├─ 光伏容量是毛刺且潜力为0 (已筛掉): {pv_tiny_and_zero}")
        print(f"    ├─ 风电容量是毛刺但潜力>0 (保留): {wind_tiny_but_has_potential}")
        print(f"    ├─ 光伏容量是毛刺但潜力>0 (保留): {pv_tiny_but_has_potential}")
        print(f"    ├─ 两端都不可行: {both_inf}")
        print(f"    └─ 系数异常大 (>= {max_reasonable_coef:.0e}): {coef_too_large}")
        
        # 保存文件
        output_path = input_path.replace('.csv', '_withPotential.csv')
        df.to_csv(output_path, index=False)
        print(f"  ✓ 已保存: {os.path.basename(output_path)}")
        
    except Exception as e:
        print(f"  ✗ 错误: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*80}")
print(f"完成! 共处理 {len(result_files)} 个文件")
print(f"{'='*80}")

## 全球计算（水资源）

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import glob
import os
import re
import xarray as xr
import geopandas as gpd

# 读取 0.25 度分辨率的风电和光伏装机潜力文件
print("正在读取风电/光伏潜力图层...")
cp_wind_025d = xr.open_dataarray(r"D:\Workspace\CP_wind_025d_GW.nc")
cp_pv_025d = xr.open_dataarray(r"D:\Workspace\CP_pv_025d_GW.nc")

# 读取水资源图层 (水量 m3)
print("正在读取水资源图层...")
water_path = r"F:\Workspace\Water\ERA5_runoff_volume_5y_avg_2021-2025.nc"
water_volume_da = None
try:
    ds_water = xr.open_dataset(water_path)
    water_var = list(ds_water.data_vars)[0]
    water_volume_da = ds_water[water_var].fillna(0)
    print(f"  已加载水资源数据: {water_var}")
except Exception as e:
    print(f"  ⚠️ 无法加载水资源文件: {e}")

# 读取 HydroBASINS Shapefile
print("正在读取 HydroBASINS 流域文件...")
basin_path = r"F:\Workspace\Water\hybas_lev04.shp"
gdf_basins = None
basin_supply_map = {}

try:
    gdf_basins = gpd.read_file(basin_path)
    if gdf_basins.crs is None:
        gdf_basins.set_crs('EPSG:4326', inplace=True)
    elif gdf_basins.crs != 'EPSG:4326':
        gdf_basins = gdf_basins.to_crs('EPSG:4326')
    print(f"  已加载流域文件，共 {len(gdf_basins)} 个流域")
    
    if water_volume_da is not None:
        print("正在将水资源汇总到流域 (这可能需要一点时间)...")
        
        # --- ⭐ 关键修复：处理水资源数据的经度问题 (0-360 -> -180-180) ---
        # ERA5 数据通常是 0-360，而 HydroBASINS 和结果文件通常是 -180-180
        # 如果不转换，西半球（美洲）的水资源点将无法落入流域多边形中
        if water_volume_da.longitude.max() > 180:
            print("  ⚠️ 检测到水资源数据经度为 0-360，正在转换为 -180-180 以匹配流域文件...")
            water_volume_da = water_volume_da.assign_coords(longitude=(((water_volume_da.longitude + 180) % 360) - 180))
            water_volume_da = water_volume_da.sortby('longitude')
        # ------------------------------------------------------------

        # 1. 转换水资源网格为 GeoDataFrame
        df_water = water_volume_da.to_dataframe().reset_index()
        df_water = df_water[df_water[water_var] > 0] # 只保留有水的网格
        
        # 空间过滤：只保留在流域边界框内的水资源点 (加速)
        minx, miny, maxx, maxy = gdf_basins.total_bounds
        # 确保 df_water 的经纬度也在 -180~180 范围内
        df_water = df_water[
            (df_water.longitude >= minx) & (df_water.longitude <= maxx) &
            (df_water.latitude >= miny) & (df_water.latitude <= maxy)
        ]
        
        gdf_water = gpd.GeoDataFrame(
            df_water,
            geometry=gpd.points_from_xy(df_water.longitude, df_water.latitude),
            crs='EPSG:4326'
        )
        
        # 2. 空间连接：水资源点 -> 流域
        # 使用 sjoin 将水资源点映射到 HYBAS_ID
        print("  正在进行空间连接 (Points -> Basins)...")
        gdf_water_joined = gpd.sjoin(gdf_water, gdf_basins[['HYBAS_ID', 'geometry']], how='inner', predicate='intersects')
        
        # 3. 汇总：按 HYBAS_ID 求和
        # 设定 20% 为可用水量
        basin_supply_series = gdf_water_joined.groupby('HYBAS_ID')[water_var].sum() * 0.2
        basin_supply_map = basin_supply_series.to_dict()
        print(f"  已计算 {len(basin_supply_map)} 个流域的水资源可用量")
        
except Exception as e:
    print(f"  ⚠️ 处理流域水资源时出错: {e}")
    import traceback
    traceback.print_exc()
    gdf_basins = None

# 获取所有 _results.csv 文件
input_dir = 'DataOutput/Used_output/'
# 修改匹配模式以包含全球文件 (假设文件名包含 results)
result_files = glob.glob(os.path.join(input_dir, '*results.csv'))

print(f"找到 {len(result_files)} 个结果文件:")
for f in result_files:
    print(f"  - {os.path.basename(f)}")

# 准备 KDTree（风光）
print("构建风光 KDTree...")
lat_centers = cp_wind_025d['latitude'].values
lon_centers = cp_wind_025d['longitude'].values
mesh_lat, mesh_lon = np.meshgrid(lat_centers, lon_centers, indexing='ij')
grid_points = np.column_stack([mesh_lat.ravel(), mesh_lon.ravel()])
tree = cKDTree(grid_points)

# 提取扁平化的潜力数据
wind_potential_flat = cp_wind_025d.values.ravel() * 1e6  # GW转kW
pv_potential_flat = cp_pv_025d.values.ravel() * 1e6

# 批量处理所有文件
for input_path in result_files:
    print(f"\n处理文件: {os.path.basename(input_path)}")
    
    try:
        # 读取数据
        df = pd.read_csv(input_path)
        original_count = len(df)
        
        # ⭐ 第一步：筛掉平准化成本为0的点
        cost_col = None
        cost_patterns = [
            r'levelized.*cost.*per.*(kg|ton)',
            r'.*cost.*per.*(kg|ton)',
        ]
        
        for pattern in cost_patterns:
            for col in df.columns:
                if re.search(pattern, col, re.IGNORECASE):
                    cost_col = col
                    break
            if cost_col:
                break
        
        if cost_col:
            df[cost_col] = pd.to_numeric(df[cost_col], errors='coerce')
            cost_zero_count = (df[cost_col] <= 0).sum() | df[cost_col].isna().sum()
            df = df[df[cost_col] > 0].copy()
            if cost_zero_count > 0:
                print(f"  ⚠️  筛掉成本为0或无效的点: {cost_zero_count}")
        else:
            print("  ⚠️ 未找到成本列，无法进行基于成本的水资源分配排序！")
        
        if len(df) == 0:
            print(f"  跳过: 筛选后无有效数据")
            continue
        
        # 匹配容量列
        wind_col = 'wind_capacity_kW' if 'wind_capacity_kW' in df.columns else 'wind_capacity'
        pv_col = 'pv_capacity_kW' if 'pv_capacity_kW' in df.columns else 'pv_capacity'
        
        if wind_col not in df.columns or pv_col not in df.columns:
            print(f"  跳过: 缺少必要的容量列")
            continue
            
        # KDTree 查询最近网格 (风光)
        data_points = df[['latitude', 'longitude']].values
        _, idx = tree.query(data_points)
        
        # 匹配风光潜力
        df['wind_potential'] = wind_potential_flat[idx]
        df['pv_potential'] = pv_potential_flat[idx]
        
        # 获取容量数据
        wind_capacity_kW = df[wind_col]
        pv_capacity_kW = df[pv_col]
        
        min_capacity_threshold = 1.0
        wind_is_tiny = wind_capacity_kW < min_capacity_threshold
        pv_is_tiny = pv_capacity_kW < min_capacity_threshold
        wind_potential_zero = df['wind_potential'] <= 0
        pv_potential_zero = df['pv_potential'] <= 0
        
        # 风电系数
        df['wind_coef'] = np.where(
            (wind_is_tiny & wind_potential_zero) | wind_potential_zero,
            np.inf,
            np.where(
                wind_is_tiny & (df['wind_potential'] > 0),
                df['wind_potential'] / min_capacity_threshold,
                df['wind_potential'] / wind_capacity_kW
            )
        )
        
        # 光伏系数
        df['pv_coef'] = np.where(
            (pv_is_tiny & pv_potential_zero) | pv_potential_zero,
            np.inf,
            np.where(
                pv_is_tiny & (df['pv_potential'] > 0),
                df['pv_potential'] / min_capacity_threshold,
                df['pv_potential'] / pv_capacity_kW
            )
        )
        
        max_reasonable_coef = 1e10

        # --- 1. 计算仅考虑风光的潜力 (Energy Only) ---
        df['energy_coef'] = df[['wind_coef', 'pv_coef']].min(axis=1)
        
        df['production_potential_ton_energy'] = np.where(
            np.isfinite(df['energy_coef']) & (df['energy_coef'] < max_reasonable_coef),
            df['energy_coef'] * 10,
            0
        )
        
        # --- 2. 基于流域的水资源分配 ---
        if gdf_basins is not None and cost_col is not None:
            print("  正在进行基于流域的水资源分配...")
            
            # 确定单位产品水耗
            water_intensity = 10.0 # 默认
            fname_lower = os.path.basename(input_path).lower()
            if 'hydrogen' in fname_lower: water_intensity = 15.0
            elif 'ammonia' in fname_lower: water_intensity = 3.0
            elif 'methanol' in fname_lower: water_intensity = 4.0
            
            # 空间连接：将项目点映射到流域
            gdf_pts = gpd.GeoDataFrame(
                df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326'
            )
            # 只保留需要的列以加速
            gdf_pts = gpd.sjoin(gdf_pts, gdf_basins[['HYBAS_ID', 'geometry']], how='left', predicate='intersects')
            
            # 将 HYBAS_ID 映射回 df
            df['HYBAS_ID'] = gdf_pts['HYBAS_ID']
            
            # 计算需水量 (m3)
            df['water_demand_m3'] = df['production_potential_ton_energy'] * water_intensity
            
            # 映射流域可用水量
            df['basin_supply_m3'] = df['HYBAS_ID'].map(basin_supply_map).fillna(0)
            
            # --- 核心分配逻辑 ---
            # 按流域分组，组内按成本排序
            df.sort_values(by=['HYBAS_ID', cost_col], ascending=[True, True], inplace=True)
            
            # 计算每个流域内的累积需水量
            df['cum_demand'] = df.groupby('HYBAS_ID')['water_demand_m3'].cumsum()
            
            # 计算分配量
            prev_cum = df['cum_demand'] - df['water_demand_m3']
            supply = df['basin_supply_m3']
            demand = df['water_demand_m3']
            
            allocated = np.where(
                prev_cum >= supply, 0,
                np.where(df['cum_demand'] <= supply, demand, supply - prev_cum)
            )
            df['allocated_water_m3'] = np.maximum(0, allocated)
            
            # 反推受水资源限制后的产量
            df['production_potential_ton'] = df['allocated_water_m3'] / water_intensity
            
            # 标记限制因素
            df['is_water_limited'] = (df['allocated_water_m3'] < df['water_demand_m3']) & (df['water_demand_m3'] > 1e-6)
            
            print(f"  水资源分配完成。强度: {water_intensity} kg/kg")
            
        else:
            print("  ⚠️ 无法进行流域分配 (缺少流域数据或成本列)，假设水资源无限...")
            df['production_potential_ton'] = df['production_potential_ton_energy']
            df['is_water_limited'] = False

        # 统计
        total_points = len(df)
        valid_energy = (df['production_potential_ton_energy'] > 0).sum()
        valid_water = (df['production_potential_ton'] > 0).sum()
        water_limited_count = df['is_water_limited'].sum()
        
        print(f"\n  潜力计算后统计:")
        print(f"    仅风光有效点: {valid_energy}")
        print(f"    风光+水有效点: {valid_water}")
        print(f"    因缺水归零的点数: {valid_energy - valid_water}")
        print(f"    受水资源限制的点数(产能降低或归零): {water_limited_count}")
        
        # 保存文件
        output_path = input_path.replace('.csv', '_withPotential.csv')
        df.to_csv(output_path, index=False)
        print(f"  ✓ 已保存: {os.path.basename(output_path)}")
        
    except Exception as e:
        print(f"  ✗ 错误: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*80}")
print(f"完成! 共处理 {len(result_files)} 个文件")
print(f"{'='*80}")

## 中国计算

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import glob
import os
import re
import xarray as xr

# 读取 0.25 度分辨率的风电和光伏装机潜力文件
cp_wind_025d = xr.open_dataarray(r"D:\Workspace\CP_wind_025d_GW.nc")
cp_pv_025d = xr.open_dataarray(r"D:\Workspace\CP_pv_025d_GW.nc")

# 获取所有 _results.csv 文件
input_dir = 'DataOutput/Used_output/'
result_files = glob.glob(os.path.join(input_dir, '*China_results.csv'))

print(f"找到 {len(result_files)} 个结果文件:")
for f in result_files:
    print(f"  - {os.path.basename(f)}")

# 准备 KDTree（只需构建一次）
lat_centers = cp_wind_025d['latitude'].values
lon_centers = cp_wind_025d['longitude'].values
mesh_lat, mesh_lon = np.meshgrid(lat_centers, lon_centers, indexing='ij')
grid_points = np.column_stack([mesh_lat.ravel(), mesh_lon.ravel()])
tree = cKDTree(grid_points)

# 提取扁平化的潜力数据
wind_potential_flat = cp_wind_025d.values.ravel() * 1e6  # GW转kW
pv_potential_flat = cp_pv_025d.values.ravel() * 1e6

# 批量处理所有文件
for input_path in result_files:
    print(f"\n处理文件: {os.path.basename(input_path)}")
    
    try:
        # 读取数据
        df = pd.read_csv(input_path)
        
        original_count = len(df)
        
        # ⭐ 第一步：筛掉平准化成本为0的点
        # 查找成本列（支持多种命名格式）
        cost_col = None
        cost_patterns = [
            r'levelized.*cost.*per.*(kg|ton)',  # levelized_xxx_cost_per_kg/ton
            r'.*cost.*per.*(kg|ton)',           # xxx_cost_per_kg/ton
        ]
        
        for pattern in cost_patterns:
            for col in df.columns:
                if re.search(pattern, col, re.IGNORECASE):
                    cost_col = col
                    break
            if cost_col:
                break
        
        if cost_col:
            print(f"  找到成本列: {cost_col}")
            
            # 转换为数值并筛选
            df[cost_col] = pd.to_numeric(df[cost_col], errors='coerce')
            cost_zero_count = (df[cost_col] <= 0).sum() | df[cost_col].isna().sum()
            
            # 只保留成本 > 0 的点
            df = df[df[cost_col] > 0].copy()
            
            if cost_zero_count > 0:
                print(f"  ⚠️  筛掉成本为0或无效的点: {cost_zero_count} ({cost_zero_count/original_count*100:.2f}%)")
                print(f"  剩余数据点: {len(df)}")
        else:
            print(f"  警告: 未找到成本列，跳过成本筛选")
        
        if len(df) == 0:
            print(f"  跳过: 筛选后无有效数据")
            continue
        
        # 灵活匹配列名：优先使用 wind_capacity_kW，否则使用 wind_capacity
        wind_col = 'wind_capacity_kW' if 'wind_capacity_kW' in df.columns else 'wind_capacity'
        pv_col = 'pv_capacity_kW' if 'pv_capacity_kW' in df.columns else 'pv_capacity'
        
        # 检查必要列是否存在
        if wind_col not in df.columns or pv_col not in df.columns:
            print(f"  跳过: 缺少必要的容量列")
            print(f"  现有列: {list(df.columns)}")
            continue
        
        print(f"  使用容量列: {wind_col}, {pv_col}")
        
        # KDTree 查询最近网格
        data_points = df[['latitude', 'longitude']].values
        _, idx = tree.query(data_points)
        
        # 匹配风光潜力
        df['wind_potential'] = wind_potential_flat[idx]
        df['pv_potential'] = pv_potential_flat[idx]
        
        # 获取容量数据
        wind_capacity_kW = df[wind_col]
        pv_capacity_kW = df[pv_col]
        
        # ⭐⭐⭐ 关键修复：只筛掉"一端是毛刺且另一端潜力为0"的点 ⭐⭐⭐
        # 设定最小有效容量阈值（1 kW）
        min_capacity_threshold = 1.0  # kW
        
        # 判断是否为毛刺容量
        wind_is_tiny = wind_capacity_kW < min_capacity_threshold
        pv_is_tiny = pv_capacity_kW < min_capacity_threshold
        
        # 判断潜力是否为0
        wind_potential_zero = df['wind_potential'] <= 0
        pv_potential_zero = df['pv_potential'] <= 0
        
        # ⭐ 核心逻辑：只在以下情况设置 coef = inf（表示该路径不可行）
        # 1. 容量是毛刺 AND 潜力为0  → 明确不可行
        # 2. 容量 >= 阈值但潜力为0    → 潜力限制
        # 3. 容量是毛刺 AND 潜力 > 0  → 保留！计算 coef
        
        # 风电系数：只有当"容量太小且潜力为0"或"潜力为0"时才设为 inf
        df['wind_coef'] = np.where(
            (wind_is_tiny & wind_potential_zero) | wind_potential_zero,
            np.inf,
            np.where(
                wind_is_tiny & (df['wind_potential'] > 0),
                df['wind_potential'] / min_capacity_threshold,  # 使用阈值避免除以极小值
                df['wind_potential'] / wind_capacity_kW
            )
        )
        
        # 光伏系数：同样逻辑
        df['pv_coef'] = np.where(
            (pv_is_tiny & pv_potential_zero) | pv_potential_zero,
            np.inf,
            np.where(
                pv_is_tiny & (df['pv_potential'] > 0),
                df['pv_potential'] / min_capacity_threshold,  # 使用阈值避免除以极小值
                df['pv_potential'] / pv_capacity_kW
            )
        )
        
        # 取最小系数（选择更限制的资源）
        min_coef = df[['wind_coef', 'pv_coef']].min(axis=1)
        
        # 设定合理的系数上限（避免异常大的值）
        max_reasonable_coef = 1e10
        
        # 计算生产潜力
        # ⚠️ 注意：所有产品（hydrogen, ammonia, methanol, FTL）的装机容量都是针对 10 吨（10,000 kg）的年产量优化的
        # min_coef 表示实际装机潜力与优化装机容量的比值
        # 因此，生产潜力 = min_coef × 10 吨
        df['production_potential_ton'] = np.where(
            np.isfinite(min_coef) & (min_coef < max_reasonable_coef),
            min_coef * 10,  # 乘以10吨（目标产量）
            0
        )
        
        # 统计信息
        total_points = len(df)
        valid_points = (df['production_potential_ton'] > 0).sum()
        filtered_points = total_points - valid_points
        
        # 详细统计各种情况
        wind_tiny_and_zero = (wind_is_tiny & wind_potential_zero).sum()
        pv_tiny_and_zero = (pv_is_tiny & pv_potential_zero).sum()
        wind_tiny_but_has_potential = (wind_is_tiny & (df['wind_potential'] > 0)).sum()
        pv_tiny_but_has_potential = (pv_is_tiny & (df['pv_potential'] > 0)).sum()
        both_inf = ((df['wind_coef'] == np.inf) & (df['pv_coef'] == np.inf)).sum()
        coef_too_large = ((min_coef >= max_reasonable_coef) & np.isfinite(min_coef)).sum()
        
        print(f"\n  潜力计算后统计:")
        print(f"    总数据点: {total_points}")
        print(f"    有效数据点: {valid_points} ({valid_points/total_points*100:.2f}%)")
        print(f"    过滤的点: {filtered_points} ({filtered_points/total_points*100:.2f}%)")
        
        print(f"\n  容量毛刺分析:")
        print(f"    ├─ 风电容量是毛刺且潜力为0 (已筛掉): {wind_tiny_and_zero}")
        print(f"    ├─ 光伏容量是毛刺且潜力为0 (已筛掉): {pv_tiny_and_zero}")
        print(f"    ├─ 风电容量是毛刺但潜力>0 (保留): {wind_tiny_but_has_potential}")
        print(f"    ├─ 光伏容量是毛刺但潜力>0 (保留): {pv_tiny_but_has_potential}")
        print(f"    ├─ 两端都不可行: {both_inf}")
        print(f"    └─ 系数异常大 (>= {max_reasonable_coef:.0e}): {coef_too_large}")
        
        # 保存文件
        output_path = input_path.replace('.csv', '_withPotential.csv')
        df.to_csv(output_path, index=False)
        print(f"  ✓ 已保存: {os.path.basename(output_path)}")
        
    except Exception as e:
        print(f"  ✗ 错误: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*80}")
print(f"完成! 共处理 {len(result_files)} 个文件")
print(f"{'='*80}")

### 水资源计算（五年平均）

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import os

# 文件路径配置
data_dir = r"F:\Workspace\Water"
filenames = [
    "ERA5_runoffrate_2021-2022.nc",
    "ERA5_runoffrate_2023.nc",
    "ERA5_runoffrate_2024-2025.nc"
]
file_paths = [os.path.join(data_dir, f) for f in filenames]
output_filename = "ERA5_runoff_volume_5y_avg_2021-2025.nc"
output_path = os.path.join(data_dir, output_filename)

try:
    print("正在加载多个数据集...")
    # 使用 open_mfdataset 读取并合并时间维度
    ds = xr.open_mfdataset(file_paths, combine='by_coords')
    print("数据集加载成功")
    
    # 自动识别变量名
    var_name = list(ds.data_vars)[0]
    print(f"处理变量: {var_name}")
    
    # 识别时间维度名称
    time_dim = 'time'
    if 'valid_time' in ds.coords:
        time_dim = 'valid_time'
    print(f"时间维度: {time_dim}")
    
    # 检查单位
    units = ds[var_name].attrs.get('units', 'unknown')
    print(f"变量单位: {units}")
    
    # 估算时间步长 (秒)
    times = ds[time_dim].values[:2]
    time_step_seconds = (times[1] - times[0]) / np.timedelta64(1, 's')
    print(f"检测到时间步长: {time_step_seconds} 秒")
    
    print("正在计算年总量...")
    # 先按年求和
    water_yearly_sum = ds[var_name].resample({time_dim: '1Y'}).sum()
    
    # 根据单位或变量名判断是否需要乘以时间步长
    is_rate = 's-1' in units or '/s' in units or 'rate' in var_name.lower() or 'rate' in filenames[0].lower()
    
    if is_rate:
        print("检测为速率数据，正在转换为总量 (Sum * TimeStep)...")
        water_yearly_total = water_yearly_sum * time_step_seconds
        
        # 处理 kg m-2 s-1 -> m
        # 1 kg/m2 = 1 mm 水深 (假设密度 1000 kg/m3)
        # 所以除以 1000 得到米
        if 'kg' in units:
            print("检测到质量单位 (kg)，转换为深度 (m) (除以水密度 1000 kg/m³)...")
            water_yearly_total = water_yearly_total / 1000.0
            water_yearly_total.attrs['units'] = 'm'
        else:
            water_yearly_total.attrs['units'] = 'm (integrated)'
    else:
        print("检测为累积数据，直接使用求和结果...")
        water_yearly_total = water_yearly_sum
        
    # 计算5年平均径流深 (m)
    print("正在计算5年平均径流深...")
    water_5y_avg_depth = water_yearly_total.mean(dim=time_dim)
    
    # --- 计算网格面积并转换为水量 (m^3) ---
    print("正在计算网格面积...")
    R = 6371000  # 地球半径 (米)
    
    # 获取纬度和经度分辨率
    lats = ds.latitude.values
    lons = ds.longitude.values
    # 假设是等经纬度网格
    dlat = np.abs(lats[1] - lats[0])
    dlon = np.abs(lons[1] - lons[0])
    
    # 转换为弧度
    rad_lats = np.deg2rad(lats)
    rad_dlon = np.deg2rad(dlon)
    rad_dlat = np.deg2rad(dlat)
    
    # 计算每个纬度带的网格面积 (m^2)
    # Area = R^2 * dlon * (sin(lat_top) - sin(lat_bottom))
    # 假设 lat 是网格中心
    lat_bnds_top = rad_lats + rad_dlat / 2
    lat_bnds_bot = rad_lats - rad_dlat / 2
    
    # 面积只与纬度有关
    cell_areas_m2 = R**2 * rad_dlon * np.abs(np.sin(lat_bnds_top) - np.sin(lat_bnds_bot))
    
    # 创建面积 DataArray
    da_area = xr.DataArray(cell_areas_m2, coords={'latitude': ds.latitude}, dims='latitude')
    
    print("正在将径流深转换为水量 (Volume = Depth * Area)...")
    water_volume_m3 = water_5y_avg_depth * da_area
    water_volume_m3.attrs['units'] = 'm^3'
    water_volume_m3.name = 'runoff_volume'
    # ------------------------------------

    # 保存为 NetCDF
    print(f"正在保存水量结果到: {output_path}")
    water_volume_m3.to_netcdf(output_path)
    print("保存成功！")

    # 绘图
    plt.figure(figsize=(12, 6))
    water_volume_m3.plot(cmap='Blues', robust=True)
    plt.title(f'5-Year Average Annual Runoff Volume (2021-2025)\nSource: {var_name} (Unit: m^3/year)')
    plt.show()
    
    print(f"计算完成。水量范围: {water_volume_m3.min().values:.2e} ~ {water_volume_m3.max().values:.2e} m^3")

except Exception as e:
    print(f"发生错误: {e}")

In [ ]:
import geopandas as gpd
import numpy as np
import os

# 定义文件路径
path_lev04 = r"F:\Workspace\Water\hybas_lev04.shp"
path_lev05 = r"F:\Workspace\Water\hybas_lev05.shp"

def analyze_basin_radius(path, level_name):
    print(f"正在读取 {level_name}: {os.path.basename(path)} ...")
    try:
        gdf = gpd.read_file(path)
        print(f"  流域数量: {len(gdf)}")
        
        # 投影到等面积投影以计算准确面积
        # 使用 World Mollweide (ESRI:54009) 适合全球面积计算
        # 如果环境不支持 ESRI 码，可以尝试 EPSG:6933 (EASE-Grid 2.0)
        target_crs = 'ESRI:54009'
        try:
            gdf_proj = gdf.to_crs(target_crs)
        except:
            print(f"  ⚠️ 无法投影到 {target_crs}，尝试使用 EPSG:6933...")
            gdf_proj = gdf.to_crs('EPSG:6933')
            
        # 计算面积 (平方米 -> 平方千米)
        areas_sqkm = gdf_proj.area / 1e6
        
        # 计算等效半径 (假设流域为圆形)
        # Area = pi * r^2  =>  r = sqrt(Area / pi)
        radii_km = np.sqrt(areas_sqkm / np.pi)
        
        mean_area = areas_sqkm.mean()
        mean_r = radii_km.mean()
        median_r = radii_km.median()
        
        print(f"  平均面积: {mean_area:.2f} km²")
        print(f"  平均等效半径: {mean_r:.2f} km")
        print(f"  中位等效半径: {median_r:.2f} km")
        print(f"  半径范围: {radii_km.min():.2f} km - {radii_km.max():.2f} km")
        
        return mean_r
    except Exception as e:
        print(f"  处理 {level_name} 时出错: {e}")
        return None

print("--- HydroBASINS 流域尺度分析 ---")
r4 = analyze_basin_radius(path_lev04, "Level 04")
print("-" * 40)
r5 = analyze_basin_radius(path_lev05, "Level 05")

if r4 and r5:
    print("-" * 40)
    print(f"比较结果: Level 04 的平均半径是 Level 05 的 {r4/r5:.2f} 倍")

## 中国计算（考虑用水）

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import glob
import os
import re
import xarray as xr
import geopandas as gpd

# 读取 0.25 度分辨率的风电和光伏装机潜力文件
print("正在读取风电/光伏潜力图层...")
cp_wind_025d = xr.open_dataarray(r"D:\Workspace\CP_wind_025d_GW.nc")
cp_pv_025d = xr.open_dataarray(r"D:\Workspace\CP_pv_025d_GW.nc")

# 读取水资源图层 (水量 m3)
print("正在读取水资源图层...")
water_path = r"F:\Workspace\Water\ERA5_runoff_volume_5y_avg_2021-2025.nc"
water_volume_da = None
try:
    ds_water = xr.open_dataset(water_path)
    water_var = list(ds_water.data_vars)[0]
    water_volume_da = ds_water[water_var].fillna(0)
    print(f"  已加载水资源数据: {water_var}")
except Exception as e:
    print(f"  ⚠️ 无法加载水资源文件: {e}")

# 读取 HydroBASINS Shapefile
print("正在读取 HydroBASINS 流域文件...")
basin_path = r"F:\Workspace\Water\hybas_lev04.shp"
gdf_basins = None
basin_supply_map = {}

try:
    gdf_basins = gpd.read_file(basin_path)
    if gdf_basins.crs is None:
        gdf_basins.set_crs('EPSG:4326', inplace=True)
    elif gdf_basins.crs != 'EPSG:4326':
        gdf_basins = gdf_basins.to_crs('EPSG:4326')
    print(f"  已加载流域文件，共 {len(gdf_basins)} 个流域")
    
    if water_volume_da is not None:
        print("正在将水资源汇总到流域 (这可能需要一点时间)...")
        # 1. 转换水资源网格为 GeoDataFrame
        df_water = water_volume_da.to_dataframe().reset_index()
        df_water = df_water[df_water[water_var] > 0] # 只保留有水的网格
        
        # 空间过滤：只保留在流域边界框内的水资源点 (加速)
        minx, miny, maxx, maxy = gdf_basins.total_bounds
        df_water = df_water[
            (df_water.longitude >= minx) & (df_water.longitude <= maxx) &
            (df_water.latitude >= miny) & (df_water.latitude <= maxy)
        ]
        
        gdf_water = gpd.GeoDataFrame(
            df_water,
            geometry=gpd.points_from_xy(df_water.longitude, df_water.latitude),
            crs='EPSG:4326'
        )
        
        # 2. 空间连接：水资源点 -> 流域
        # 使用 sjoin 将水资源点映射到 HYBAS_ID
        gdf_water_joined = gpd.sjoin(gdf_water, gdf_basins[['HYBAS_ID', 'geometry']], how='inner', predicate='intersects')
        
        # 3. 汇总：按 HYBAS_ID 求和
        # 设定 20% 为可用水量
        basin_supply_series = gdf_water_joined.groupby('HYBAS_ID')[water_var].sum() * 0.2
        basin_supply_map = basin_supply_series.to_dict()
        print(f"  已计算 {len(basin_supply_map)} 个流域的水资源可用量")
        
except Exception as e:
    print(f"  ⚠️ 处理流域水资源时出错: {e}")
    gdf_basins = None

# 获取所有 _results.csv 文件
input_dir = 'DataOutput/Used_output/'
result_files = glob.glob(os.path.join(input_dir, '*China_results.csv'))

print(f"找到 {len(result_files)} 个结果文件:")
for f in result_files:
    print(f"  - {os.path.basename(f)}")

# 准备 KDTree（风光）
print("构建风光 KDTree...")
lat_centers = cp_wind_025d['latitude'].values
lon_centers = cp_wind_025d['longitude'].values
mesh_lat, mesh_lon = np.meshgrid(lat_centers, lon_centers, indexing='ij')
grid_points = np.column_stack([mesh_lat.ravel(), mesh_lon.ravel()])
tree = cKDTree(grid_points)

# 提取扁平化的潜力数据
wind_potential_flat = cp_wind_025d.values.ravel() * 1e6  # GW转kW
pv_potential_flat = cp_pv_025d.values.ravel() * 1e6

# 批量处理所有文件
for input_path in result_files:
    print(f"\n处理文件: {os.path.basename(input_path)}")
    
    try:
        # 读取数据
        df = pd.read_csv(input_path)
        original_count = len(df)
        
        # ⭐ 第一步：筛掉平准化成本为0的点
        cost_col = None
        cost_patterns = [
            r'levelized.*cost.*per.*(kg|ton)',
            r'.*cost.*per.*(kg|ton)',
        ]
        
        for pattern in cost_patterns:
            for col in df.columns:
                if re.search(pattern, col, re.IGNORECASE):
                    cost_col = col
                    break
            if cost_col:
                break
        
        if cost_col:
            df[cost_col] = pd.to_numeric(df[cost_col], errors='coerce')
            cost_zero_count = (df[cost_col] <= 0).sum() | df[cost_col].isna().sum()
            df = df[df[cost_col] > 0].copy()
            if cost_zero_count > 0:
                print(f"  ⚠️  筛掉成本为0或无效的点: {cost_zero_count}")
        else:
            print("  ⚠️ 未找到成本列，无法进行基于成本的水资源分配排序！")
        
        if len(df) == 0:
            print(f"  跳过: 筛选后无有效数据")
            continue
        
        # 匹配容量列
        wind_col = 'wind_capacity_kW' if 'wind_capacity_kW' in df.columns else 'wind_capacity'
        pv_col = 'pv_capacity_kW' if 'pv_capacity_kW' in df.columns else 'pv_capacity'
        
        if wind_col not in df.columns or pv_col not in df.columns:
            print(f"  跳过: 缺少必要的容量列")
            continue
            
        # KDTree 查询最近网格 (风光)
        data_points = df[['latitude', 'longitude']].values
        _, idx = tree.query(data_points)
        
        # 匹配风光潜力
        df['wind_potential'] = wind_potential_flat[idx]
        df['pv_potential'] = pv_potential_flat[idx]
        
        # 获取容量数据
        wind_capacity_kW = df[wind_col]
        pv_capacity_kW = df[pv_col]
        
        min_capacity_threshold = 1.0
        wind_is_tiny = wind_capacity_kW < min_capacity_threshold
        pv_is_tiny = pv_capacity_kW < min_capacity_threshold
        wind_potential_zero = df['wind_potential'] <= 0
        pv_potential_zero = df['pv_potential'] <= 0
        
        # 风电系数
        df['wind_coef'] = np.where(
            (wind_is_tiny & wind_potential_zero) | wind_potential_zero,
            np.inf,
            np.where(
                wind_is_tiny & (df['wind_potential'] > 0),
                df['wind_potential'] / min_capacity_threshold,
                df['wind_potential'] / wind_capacity_kW
            )
        )
        
        # 光伏系数
        df['pv_coef'] = np.where(
            (pv_is_tiny & pv_potential_zero) | pv_potential_zero,
            np.inf,
            np.where(
                pv_is_tiny & (df['pv_potential'] > 0),
                df['pv_potential'] / min_capacity_threshold,
                df['pv_potential'] / pv_capacity_kW
            )
        )
        
        max_reasonable_coef = 1e10

        # --- 1. 计算仅考虑风光的潜力 (Energy Only) ---
        df['energy_coef'] = df[['wind_coef', 'pv_coef']].min(axis=1)
        
        df['production_potential_ton_energy'] = np.where(
            np.isfinite(df['energy_coef']) & (df['energy_coef'] < max_reasonable_coef),
            df['energy_coef'] * 10,
            0
        )
        
        # --- 2. 基于流域的水资源分配 ---
        if gdf_basins is not None and cost_col is not None:
            print("  正在进行基于流域的水资源分配...")
            
            # 确定单位产品水耗
            water_intensity = 10.0 # 默认
            fname_lower = os.path.basename(input_path).lower()
            if 'hydrogen' in fname_lower: water_intensity = 15.0
            elif 'ammonia' in fname_lower: water_intensity = 2.64
            elif 'methanol' in fname_lower: water_intensity = 2.88
            
            # 空间连接：将项目点映射到流域
            gdf_pts = gpd.GeoDataFrame(
                df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326'
            )
            # 只保留需要的列以加速
            gdf_pts = gpd.sjoin(gdf_pts, gdf_basins[['HYBAS_ID', 'geometry']], how='left', predicate='intersects')
            
            # 将 HYBAS_ID 映射回 df (注意索引对齐)
            # sjoin 可能会改变索引或顺序，最好使用 merge 或直接赋值如果索引未变
            # 这里假设 sjoin 后索引保留原 df 的索引 (how='left')
            df['HYBAS_ID'] = gdf_pts['HYBAS_ID']
            
            # 计算需水量 (m3)
            df['water_demand_m3'] = df['production_potential_ton_energy'] * water_intensity
            
            # 映射流域可用水量
            df['basin_supply_m3'] = df['HYBAS_ID'].map(basin_supply_map).fillna(0)
            
            # --- 核心分配逻辑 ---
            # 按流域分组，组内按成本排序
            # 为了高效，我们先对整个 df 排序
            df.sort_values(by=['HYBAS_ID', cost_col], ascending=[True, True], inplace=True)
            
            # 计算每个流域内的累积需水量
            df['cum_demand'] = df.groupby('HYBAS_ID')['water_demand_m3'].cumsum()
            
            # 计算分配量
            # 如果 (累积需求 - 当前需求) >= 供给，则分配 0
            # 如果 累积需求 <= 供给，则分配 当前需求
            # 否则，分配 (供给 - (累积需求 - 当前需求))
            
            prev_cum = df['cum_demand'] - df['water_demand_m3']
            supply = df['basin_supply_m3']
            demand = df['water_demand_m3']
            
            allocated = np.where(
                prev_cum >= supply, 0,
                np.where(df['cum_demand'] <= supply, demand, supply - prev_cum)
            )
            # 确保非负 (浮点误差)
            df['allocated_water_m3'] = np.maximum(0, allocated)
            
            # 反推受水资源限制后的产量
            # 产量 = 分配水量 / 水耗强度
            df['production_potential_ton'] = df['allocated_water_m3'] / water_intensity
            
            # 标记限制因素
            # 如果 分配水量 < 需求水量 (且需求 > 0)，则受水限制
            df['is_water_limited'] = (df['allocated_water_m3'] < df['water_demand_m3']) & (df['water_demand_m3'] > 1e-6)
            
            print(f"  水资源分配完成。强度: {water_intensity} kg/kg")
            
        else:
            print("  ⚠️ 无法进行流域分配 (缺少流域数据或成本列)，假设水资源无限...")
            df['production_potential_ton'] = df['production_potential_ton_energy']
            df['is_water_limited'] = False

        # 统计
        total_points = len(df)
        valid_energy = (df['production_potential_ton_energy'] > 0).sum()
        valid_water = (df['production_potential_ton'] > 0).sum()
        water_limited_count = df['is_water_limited'].sum()
        
        print(f"\n  潜力计算后统计:")
        print(f"    仅风光有效点: {valid_energy}")
        print(f"    风光+水有效点: {valid_water}")
        print(f"    因缺水归零的点数: {valid_energy - valid_water}")
        print(f"    受水资源限制的点数(产能降低或归零): {water_limited_count}")
        
        # 保存文件
        output_path = input_path.replace('.csv', '_withPotential.csv')
        df.to_csv(output_path, index=False)
        print(f"  ✓ 已保存: {os.path.basename(output_path)}")
        
    except Exception as e:
        print(f"  ✗ 错误: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*80}")
print(f"完成! 共处理 {len(result_files)} 个文件")
print(f"{'='*80}")

## 氢能全球分段填色地图（离散色阶）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import os
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

import cartopy.io.shapereader as shpreader
from cartopy.feature import ShapelyFeature

# =========================
# 氢能全球分段填色地图（按平准化成本 LCOH 绘图）
# =========================

# 基础字体设置，偏 Nature 期刊
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 8

# 查找氢能数据文件
input_dir = 'DataOutput/Used_output/'
hydrogen_files_all = sorted(glob.glob(os.path.join(input_dir, '*hydrogen*_withPotential.csv')))

keep_years = ['2030', '2040', '2050', '2100']

# 先筛选出 2030、2050 和“无年份”的情景
files_2030 = [f for f in hydrogen_files_all if '2030' in f]
files_2050 = [f for f in hydrogen_files_all if '2050' in f]
files_noyear = [f for f in hydrogen_files_all
                if all(y not in f for y in keep_years)]

# 再按指定顺序组合：2030 → 无年份 → 2050
hydrogen_files = sorted(files_2030) + sorted(files_noyear) + sorted(files_2050)

print(f"找到 {len(hydrogen_files)} 个氢能数据文件（按 2030 → 无年份 → 2050 排序）\n")

# ========= 1. 根据“平准化成本”自然断点分段 =========
cost_boundaries = [0.9, 1.2, 1.6, 2, 2.5, 3, 5]

n_bins = len(cost_boundaries) - 1

# ========= 2. 更清晰的离散蓝系配色（深→浅） =========
colors_list = [
    '#08306b',  # 深蓝
    '#08519c',
    '#2171b5',
    '#6baed6',
    '#9ecae1',
    '#c6dbef'
][:n_bins]

cmap = ListedColormap(colors_list)
norm = BoundaryNorm(cost_boundaries, cmap.N, clip=True)

# ========= 使用自定义世界地图 shapefile =========
shp_path = r'F:\Workspace\世界国家_new.shp'   # 用原始字符串，避免反斜杠转义
reader = shpreader.Reader(shp_path)

# 读取几何对象
world_geoms = list(reader.geometries())

# 做成一个 Cartopy Feature
world_feature = ShapelyFeature(
    world_geoms,
    ccrs.PlateCarree(),          # 如果你的 shp 是经纬度坐标，这里用 PlateCarree
    facecolor='#f5f5f5',         # 类似之前的陆地区颜色
    edgecolor='black',
    linewidth=0.3
)


# ========= 3. 主图：三张地图（竖排 3×1） =========
n_files = len(hydrogen_files)
if n_files == 0:
    print("未找到氢能数据文件！")
else:
    # 固定按三张图竖排设计
    nrows, ncols = 3, 1
    figsize = (10, 12)

    fig = plt.figure(figsize=figsize)

    for idx, file_path in enumerate(hydrogen_files[:3], 1):
        file_name = os.path.basename(file_path).replace('combined_', '').replace('_withPotential.csv', '')
        print(f"处理: {file_name}")

        # 读取数据
        df = pd.read_csv(file_path)

        # 查找成本列和潜力列
        cost_col = None
        for col in df.columns:
            if 'hydrogen' in col.lower() and 'cost' in col.lower() and 'kg' in col.lower():
                cost_col = col
                break

        potential_col = 'production_potential_ton'

        if cost_col not in df.columns or potential_col not in df.columns:
            print(f"  跳过: 缺少必要列 ({cost_col}, {potential_col})")
            continue

        # 过滤有效数据
        df_valid = df[(df[cost_col] > 0) & (df[potential_col] > 0)].copy()
        print(f"  有效数据点: {len(df_valid):,}")

        if len(df_valid) == 0:
            print(f"  无有效数据，跳过\n")
            continue

        # 转换潜力单位为 Mt（仅用于统计打印）
        df_valid['potential_Mt'] = df_valid[potential_col] / 1e6
        total_potential = df_valid['potential_Mt'].sum()
        mean_cost = df_valid[cost_col].mean()

        print(f"  总潜力: {total_potential:.2f} Mt")
        print(f"  平均成本: ${mean_cost:.2f}/kg")

        print(f"  各成本区间分布:")
        for i in range(len(cost_boundaries)-1):
            in_bin = (df_valid[cost_col] >= cost_boundaries[i]) & (df_valid[cost_col] < cost_boundaries[i+1])
            count = in_bin.sum()
            pot_sum = df_valid.loc[in_bin, 'potential_Mt'].sum()
            if count > 0:
                print(f"    [{cost_boundaries[i]:.1f} - {cost_boundaries[i+1]:.1f}] $/kg: "
                      f"{count:,} 点, 潜力合计 {pot_sum:.2f} Mt")

        # 创建子图
        ax = fig.add_subplot(nrows, ncols, idx, projection=ccrs.PlateCarree())

        # 干净背景：自定义世界地图 + 白色海洋
        ax.set_facecolor('white')          # 海洋底色
        ax.add_feature(world_feature, zorder = 1)      # 自己的世界地图 shp
        ax.set_global()

        # 点大小和潜力相关（适当放大，便于观看）
        s_min, s_max = 0.3, 0.3
        pot = df_valid['potential_Mt']
        pot_norm = (np.log10(pot + 1e-6) - np.log10(pot.min() + 1e-6)) / max(
            np.log10(pot.max() + 1e-6) - np.log10(pot.min() + 1e-6), 1e-6
        )
        size = s_min + (s_max - s_min) * pot_norm

        scatter = ax.scatter(
            df_valid['longitude'],
            df_valid['latitude'],
            c=df_valid[cost_col],
            cmap=cmap,
            norm=norm,
            s=size,
            alpha=0.85,
            transform=ccrs.PlateCarree(),
            linewidths=0,
            edgecolors='none',
            zorder=3              # ⭐ 保证在 polygon 上面
        )

        # # 仅保留淡淡网格线，无标题、无 total potential 文本
        # gl = ax.gridlines(
        #     draw_labels=False,
        #     linewidth=0.25,
        #     color='gray',
        #     alpha=0.4,
        #     linestyle='--'
        # )

        print(f"  ✓ 完成\n")

        ax.set_title(" ", pad=20)   # 空字符串作为标题，pad 控制与图之间的距离

    # 让地图尽量占满画布（无图例，无标题）
    fig.tight_layout(rect=[0.02, 0.02, 0.98, 0.98])

    plt.savefig('04 Potenial Mapping/hydrogen_global_costmap_discrete.png', dpi=300, bbox_inches='tight')
    plt.savefig('04 Potenial Mapping/hydrogen_global_costmap_discrete.pdf', bbox_inches='tight')
    plt.show()

    print("="*80)
    print("分段成本地图已保存:")
    print("  - hydrogen_global_costmap_discrete.png")
    print("  - hydrogen_global_costmap_discrete.pdf")
    print("="*80)

    # ========= 4. 单独绘制图例 =========
    legend_labels = []
    for i in range(len(cost_boundaries)-1):
        label = f'{cost_boundaries[i]:.1f}-{cost_boundaries[i+1]:.1f} $/kg'
        legend_labels.append(label)

    legend_patches = [
        mpatches.Patch(color=colors_list[i], label=legend_labels[i])
        for i in range(len(colors_list))
    ]

    fig_leg = plt.figure(figsize=(1, 4))
    ax_leg = fig_leg.add_subplot(111)
    ax_leg.axis('off')

    ncols_leg = int(np.ceil(len(legend_patches) / 6))  # 两行图例
    leg = fig_leg.legend(
        handles=legend_patches,
        loc='center',
        ncol=ncols_leg,
        title='平准化氢气成本 (LCOH)',
        frameon=False,
        fontsize=8,
        title_fontsize=9,
        columnspacing=2.0,      # 增加列之间的间距(默认2.0)
        handletextpad=0.8,      # 增加图例标记与文字间距(默认0.8)
        labelspacing=2.0,       # 增加垂直行间距(默认0.5)
        borderaxespad=1.0       # 增加图例与轴边界的间距(默认0.5)
    )

    fig_leg.tight_layout()
    fig_leg.savefig('04 Potenial Mapping/hydrogen_global_costmap_legend.png', dpi=300, bbox_inches='tight')
    fig_leg.savefig('04 Potenial Mapping/hydrogen_global_costmap_legend.pdf', bbox_inches='tight')
    plt.show()

    print("图例已单独保存：")
    print("  - hydrogen_global_costmap_legend.png")
    print("  - hydrogen_global_costmap_legend.pdf")


## 中国填色地图

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import os
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

import cartopy.io.shapereader as shpreader
from cartopy.feature import ShapelyFeature

# =========================
# 氢能全球分段填色地图（按平准化成本 LCOH 绘图）
# =========================

# 基础字体设置，偏 Nature 期刊
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 8

# 查找氢能数据文件
input_dir = 'DataOutput/Used_output/China'
hydrogen_files_all = sorted(glob.glob(os.path.join(input_dir, '*ammonia*_withPotential.csv')))

keep_years = ['2030', '2040', '2050', '2100']

# 先筛选出 2030、2050 和“无年份”的情景
files_2030 = [f for f in hydrogen_files_all if '2030' in f]
files_2050 = [f for f in hydrogen_files_all if '2050' in f]
files_noyear = [f for f in hydrogen_files_all
                if all(y not in f for y in keep_years)]

# 再按指定顺序组合：2030 → 无年份 → 2050
hydrogen_files = sorted(files_2030) + sorted(files_noyear) + sorted(files_2050)

print(f"找到 {len(hydrogen_files)} 个氢能数据文件（按 2030 → 无年份 → 2050 排序）\n")

# ========= 1. 根据“平准化成本”自然断点分段 =========
cost_boundaries = [100, 150, 200, 250, 300, 400, 500]

n_bins = len(cost_boundaries) - 1

# ========= 2. 更清晰的离散蓝系配色（深→浅） =========
colors_list = [
    '#08306b',  # 深蓝
    '#08519c',
    '#2171b5',
    '#6baed6',
    '#9ecae1',
    '#c6dbef'
][:n_bins]

cmap = ListedColormap(colors_list)
norm = BoundaryNorm(cost_boundaries, cmap.N, clip=True)

# ========= 使用自定义世界地图 shapefile =========
shp_path = r'F:\Workspace\世界国家_new.shp'   # 用原始字符串，避免反斜杠转义
reader = shpreader.Reader(shp_path)

# 读取几何对象
world_geoms = list(reader.geometries())

# 做成一个 Cartopy Feature
world_feature = ShapelyFeature(
    world_geoms,
    ccrs.PlateCarree(),          # 如果你的 shp 是经纬度坐标，这里用 PlateCarree
    facecolor='#f5f5f5',         # 类似之前的陆地区颜色
    edgecolor='black',
    linewidth=0.3
)


# ========= 3. 主图：三张地图（竖排 3×1） =========
n_files = len(hydrogen_files)
if n_files == 0:
    print("未找到氢能数据文件！")
else:
    # 固定按三张图竖排设计
    nrows, ncols = 3, 1
    figsize = (10, 12)

    fig = plt.figure(figsize=figsize)

    for idx, file_path in enumerate(hydrogen_files[:3], 1):
        print(file_path)
        file_name = os.path.basename(file_path).replace('combined_', '').replace('_withPotential.csv', '')
        print(f"处理: {file_name}")

        # 读取数据
        df = pd.read_csv(file_path)

        # 查找成本列和潜力列
        cost_col = None
        for col in df.columns:
            if 'ammonia' in col.lower() and 'cost' in col.lower() and 'ton' in col.lower():
                cost_col = col
                break

        potential_col = 'production_potential_ton'

        if cost_col not in df.columns or potential_col not in df.columns:
            print(f"  跳过: 缺少必要列 ({cost_col}, {potential_col})")
            continue

        # 过滤有效数据
        df_valid = df[(df[cost_col] > 0) & (df[potential_col] > 0)].copy()
        print(f"  有效数据点: {len(df_valid):,}")

        if len(df_valid) == 0:
            print(f"  无有效数据，跳过\n")
            continue

        # 转换潜力单位为 Mt（仅用于统计打印）
        df_valid['potential_Mt'] = df_valid[potential_col] / 1e6
        total_potential = df_valid['potential_Mt'].sum()
        mean_cost = df_valid[cost_col].mean()

        print(f"  总潜力: {total_potential:.2f} Mt")
        print(f"  平均成本: ${mean_cost:.2f}/kg")

        print(f"  各成本区间分布:")
        for i in range(len(cost_boundaries)-1):
            in_bin = (df_valid[cost_col] >= cost_boundaries[i]) & (df_valid[cost_col] < cost_boundaries[i+1])
            count = in_bin.sum()
            pot_sum = df_valid.loc[in_bin, 'potential_Mt'].sum()
            if count > 0:
                print(f"    [{cost_boundaries[i]:.1f} - {cost_boundaries[i+1]:.1f}] $/kg: "
                      f"{count:,} 点, 潜力合计 {pot_sum:.2f} Mt")

        # 创建子图
        ax = fig.add_subplot(nrows, ncols, idx, projection=ccrs.PlateCarree())

        # 干净背景：自定义世界地图 + 白色海洋
        ax.set_facecolor('white')          # 海洋底色
        ax.add_feature(world_feature, zorder = 1)      # 自己的世界地图 shp
        ax.set_global()

        # 点大小和潜力相关（适当放大，便于观看）
        s_min, s_max = 0.3, 0.3
        pot = df_valid['potential_Mt']
        pot_norm = (np.log10(pot + 1e-6) - np.log10(pot.min() + 1e-6)) / max(
            np.log10(pot.max() + 1e-6) - np.log10(pot.min() + 1e-6), 1e-6
        )
        size = s_min + (s_max - s_min) * pot_norm

        scatter = ax.scatter(
            df_valid['longitude'],
            df_valid['latitude'],
            c=df_valid[cost_col],
            cmap=cmap,
            norm=norm,
            s=size,
            alpha=0.85,
            transform=ccrs.PlateCarree(),
            linewidths=0,
            edgecolors='none',
            zorder=3              # ⭐ 保证在 polygon 上面
        )

        # # 仅保留淡淡网格线，无标题、无 total potential 文本
        # gl = ax.gridlines(
        #     draw_labels=False,
        #     linewidth=0.25,
        #     color='gray',
        #     alpha=0.4,
        #     linestyle='--'
        # )

        print(f"  ✓ 完成\n")

        ax.set_title(" ", pad=20)   # 空字符串作为标题，pad 控制与图之间的距离

    # 让地图尽量占满画布（无图例，无标题）
    fig.tight_layout(rect=[0.02, 0.02, 0.98, 0.98])

    plt.savefig('04 Potenial Mapping/ammonia_China_costmap_discrete.png', dpi=300, bbox_inches='tight')
    plt.savefig('04 Potenial Mapping/ammonia_China_costmap_discrete.pdf', bbox_inches='tight')
    plt.show()

    print("="*80)
    print("分段成本地图已保存:")
    print("  - hydrogen_global_costmap_discrete.png")
    print("  - hydrogen_global_costmap_discrete.pdf")
    print("="*80)

    # ========= 4. 单独绘制图例 =========
    legend_labels = []
    for i in range(len(cost_boundaries)-1):
        label = f'{cost_boundaries[i]:.1f}-{cost_boundaries[i+1]:.1f} $/kg'
        legend_labels.append(label)

    legend_patches = [
        mpatches.Patch(color=colors_list[i], label=legend_labels[i])
        for i in range(len(colors_list))
    ]

    fig_leg = plt.figure(figsize=(1, 4))
    ax_leg = fig_leg.add_subplot(111)
    ax_leg.axis('off')

    ncols_leg = int(np.ceil(len(legend_patches) / 6))  # 两行图例
    leg = fig_leg.legend(
        handles=legend_patches,
        loc='center',
        ncol=ncols_leg,
        title='平准化氢气成本 (LCOH)',
        frameon=False,
        fontsize=8,
        title_fontsize=9,
        columnspacing=2.0,      # 增加列之间的间距(默认2.0)
        handletextpad=0.8,      # 增加图例标记与文字间距(默认0.8)
        labelspacing=2.0,       # 增加垂直行间距(默认0.5)
        borderaxespad=1.0       # 增加图例与轴边界的间距(默认0.5)
    )

    fig_leg.tight_layout()
    fig_leg.savefig('04 Potenial Mapping/ammonia_China_costmap_legend.png', dpi=300, bbox_inches='tight')
    fig_leg.savefig('04 Potenial Mapping/ammonia_China_costmap_legend.pdf', bbox_inches='tight')
    plt.show()

    print("图例已单独保存：")
    print("  - hydrogen_global_costmap_legend.png")
    print("  - hydrogen_global_costmap_legend.pdf")


In [ ]:
    # ========= 4. 单独绘制图例 =========
    legend_labels = []
    for i in range(len(cost_boundaries)-1):
        label = f'{cost_boundaries[i]:.1f}-{cost_boundaries[i+1]:.1f} $/kg'
        legend_labels.append(label)

    legend_patches = [
        mpatches.Patch(color=colors_list[i], label=legend_labels[i])
        for i in range(len(colors_list))
    ]

    fig_leg = plt.figure(figsize=(1, 4))
    ax_leg = fig_leg.add_subplot(111)
    ax_leg.axis('off')

    ncols_leg = int(np.ceil(len(legend_patches) / 6))  # 两行图例
    leg = fig_leg.legend(
        handles=legend_patches,
        loc='center',
        ncol=ncols_leg,
        title='平准化氢气成本 (LCOH)',
        frameon=False,
        fontsize=16,
        title_fontsize=18,
        columnspacing=2.0,      # 增加列之间的间距(默认2.0)
        handletextpad=0.8,      # 增加图例标记与文字间距(默认0.8)
        labelspacing=2.0,       # 增加垂直行间距(默认0.5)
        borderaxespad=1.0       # 增加图例与轴边界的间距(默认0.5)
    )

    fig_leg.tight_layout()
    fig_leg.savefig('04 Potenial Mapping/hydrogen_global_costmap_legend.png', dpi=300, bbox_inches='tight')
    fig_leg.savefig('04 Potenial Mapping/hydrogen_global_costmap_legend.pdf', bbox_inches='tight')
    plt.show()

    print("图例已单独保存：")
    print("  - hydrogen_global_costmap_legend.png")
    print("  - hydrogen_global_costmap_legend.pdf")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import re

# =========================
# Matplotlib/Nature风格
# =========================
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 修正拼写
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['xtick.major.width'] = 1.2
plt.rcParams['ytick.major.width'] = 1.2
plt.rcParams['xtick.major.size'] = 4
plt.rcParams['ytick.major.size'] = 4

# =========================
# 工具函数
# =========================
def find_product_from_filename(fn: str) -> str:
    s = fn.lower()
    if 'ammonia' in s or 'nh3' in s:
        return 'ammonia'
    if 'methanol' in s or 'meoh' in s:
        return 'methanol'
    if 'ftl' in s or 'ft-liquids' in s or 'fischer' in s:
        return 'ftl'
    if 'hydrogen' in s or 'h2' in s:
        return 'hydrogen'
    return 'product'

def find_cost_column(df: pd.DataFrame) -> str | None:
    """
    在列名中用正则寻找 levelized_(ammonia|methanol|ftl|hydrogen).*cost.*per.*(ton|kg)
    不区分大小写；找到了就返回列名，否则返回 None
    """
    pattern = re.compile(r'levelized_(?:ammonia|methanol|ftl|hydrogen).*cost.*per.*(?:ton|kg)', re.IGNORECASE)
    for col in df.columns:
        if pattern.search(col):
            return col
    # 常见备选名（万一没有严格遵循上面命名）
    fallbacks = [
        'levelized_ammonia_cost_per_ton', 'levelized_methanol_cost_per_ton', 'levelized_ftl_cost_per_ton',
        'ammonia_cost_per_ton', 'methanol_cost_per_ton', 'ftl_cost_per_ton',
        'levelized_hydrogen_cost_per_kg', 'hydrogen_cost_per_kg', 'levelized_hydrogen_cost_per_ton', 'hydrogen_cost_per_ton'
    ]
    for col in fallbacks:
        if col in df.columns:
            return col
    return None

def find_potential_Mt(df: pd.DataFrame, product: str) -> pd.Series | None:
    """
    返回"以 Mt 为单位"的潜力序列：
    - 若有 production_potential_Mt 直接用
    - 若有 production_potential_ton 则 /1e6
    - 若有 production_potential_kg (hydrogen) 则 /1e9
    - 若有 production_potential 则根据产品类型判断单位
    """
    if 'production_potential_Mt' in df.columns:
        return df['production_potential_Mt']
    if 'production_potential_ton' in df.columns:
        return df['production_potential_ton'] / 1e6
    if 'production_potential_kg' in df.columns:
        # 针对 hydrogen: kg -> Mt
        return df['production_potential_kg'] / 1e9
    if 'production_potential' in df.columns:
        # 根据产品类型判断单位
        if product == 'hydrogen':
            # 假设 hydrogen 的 production_potential 是 kg 单位
            return df['production_potential'] / 1e9
        else:
            # 其他产品假设单位为 ton
            return df['production_potential'] / 1e6
    return None

# =========================
# 数据与颜色
# =========================
input_dir = 'DataOutput/Used_output/'
potential_files = sorted(glob.glob(os.path.join(input_dir, '*_withPotential.csv')))

print(f"找到 {len(potential_files)} 个潜力文件")

colors = ['#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', 
          '#8491B4', '#91D1C2', '#DC0000', '#7E6148', '#B09C85',
          '#E18727', '#FFDC91']

# =========================
# 画布
# =========================
fig, axes = plt.subplots(3, 5, figsize=(16, 16))
axes = axes.flatten()

for idx, input_path in enumerate(potential_files[:14]):  # 最多12张子图
    ax = axes[idx]
    title_stub = os.path.basename(input_path).replace('combined_', '').replace('_withPotential.csv', '')
    product = find_product_from_filename(title_stub)

    try:
        df = pd.read_csv(input_path)
    except Exception as e:
        ax.text(0.5, 0.5, f'读取失败:\n{e}', ha='center', va='center', fontsize=10)
        ax.set_title(title_stub, fontsize=9)
        continue

    # 找成本列 & 潜力列（Mt）
    cost_col = find_cost_column(df)
    pot_Mt = find_potential_Mt(df, product)

    if cost_col is None or pot_Mt is None:
        missing = []
        if cost_col is None:
            missing.append('成本列(levelized_*_cost_per_ton/kg)')
        if pot_Mt is None:
            missing.append('潜力列(production_potential_Mt 或 *_ton/*_kg)')
        ax.text(0.5, 0.5, '无有效数据：\n' + ' / '.join(missing), ha='center', va='center', fontsize=10)
        ax.set_title(title_stub, fontsize=9)
        continue

    # 清洗数据
    tmp = df.copy()
    tmp = tmp.assign(
        cost_per_ton = pd.to_numeric(tmp[cost_col], errors='coerce'),
        potential_Mt = pd.to_numeric(pot_Mt, errors='coerce')
    )
    tmp = tmp[np.isfinite(tmp['cost_per_ton']) & np.isfinite(tmp['potential_Mt']) & (tmp['potential_Mt'] > 0)]

    if len(tmp) == 0:
        ax.text(0.5, 0.5, '无有效数据', ha='center', va='center', fontsize=12)
        ax.set_title(title_stub, fontsize=9)
        continue

    # 按成本升序 & 计算累积潜力
    tmp = tmp.sort_values('cost_per_ton')
    tmp['cumulative_potential_Mt'] = tmp['potential_Mt'].cumsum()

    # 画供应曲线
    ax.plot(tmp['cumulative_potential_Mt'], tmp['cost_per_ton'],
            color=colors[idx % len(colors)], linewidth=2, alpha=0.9)

    # 标题与轴标签
    prod_label_map = {
        'ammonia': '氨',
        'methanol': '甲醇',
        'ftl': '费托液体燃料',
        'hydrogen': '氢气'
    }
    prod_cn = prod_label_map.get(product, '产品')
    
    # 根据产品类型调整Y轴标签单位
    if product == 'hydrogen':
        y_label = f'{prod_cn}平准化成本 ($/kg)'
    else:
        y_label = f'{prod_cn}平准化成本 ($/吨)'
    
    ax.set_title(f'{title_stub}', fontsize=9, pad=8)
    ax.set_xlabel(f'累积{prod_cn}生产潜力 (Mt)', fontsize=9)
    ax.set_ylabel(y_label, fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.8)
    ax.set_facecolor('#FAFAFA')

    # 角标信息
    total_potential = tmp['cumulative_potential_Mt'].iloc[-1]
    ax.text(0.95, 0.95, f'{total_potential:.1f} Mt',
            transform=ax.transAxes, fontsize=8, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none'))

# 隐藏多余子图
for idx in range(len(potential_files), 9):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('04 Potenial Mapping/supply_curves_all_scenarios.png', dpi=300, bbox_inches='tight')
plt.savefig('04 Potenial Mapping/supply_curves_all_scenarios.pdf', bbox_inches='tight')
plt.show()

print("\n图表已保存为 supply_curves_all_scenarios.png 和 .pdf")

## 氢能供应曲线单独绘制

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import matplotlib.patches as mpatches
from collections import defaultdict
import matplotlib.colors as mcolors

# ===============================
# Matplotlib 风格
# ===============================
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 14
plt.rcParams['axes.linewidth'] = 1.4
plt.rcParams['lines.linewidth'] = 2.0

# ===============================
# 英文 → 中文映射
# ===============================
CONTINENT_ZH = {
    "Asia": "亚洲",
    "Africa&Middle East": "非洲与中东",
    "North America": "北美",
    "South America": "南美",
    "Oceania": "大洋洲",
    "Europe": "欧洲",
    "Other": "其他",
}

REGION_NAME_ZH = {
    "EU-15": "欧盟-15","Canada": "加拿大","USA": "美国","Central America": "中美洲",
    "Mexico": "墨西哥","Southeast Asia": "东南亚","Australia_NZ": "澳大利亚与新西兰",
    "#N/A": "无","Africa_N": "北非","Africa_W": "西非","Africa_E": "东非","Africa_S": "南部非洲",
    "South Africa": "南非","South America_N": "南美洲北部","South America_S": "南美洲南部",
    "Brazil": "巴西","Argentina": "阿根廷","European FTA": "欧洲自由贸易联盟","Russia": "俄罗斯",
    "EU-12": "欧盟-12","Europe_Eastern": "东欧","Europe_Non_EU": "欧洲（非欧盟）",
    "Central Asia": "中亚","Middle East": "中东","South Asia": "南亚","Japan": "日本",
    "South Korea": "韩国","India": "印度","Indonesia": "印度尼西亚","China": "中国","Unknown": "未知",
}

# ===============================
# 区域→大洲映射与顺序
# ===============================
REGION_TO_CONTINENT = {
    "EU-15": "Europe","Canada": "North America","USA": "North America",
    "Central America": "South America","Mexico": "North America",
    "Southeast Asia": "Asia","Australia_NZ": "Oceania","#N/A": "Other",
    "Africa_N": "Africa&Middle East","Africa_W": "Africa&Middle East",
    "Africa_E": "Africa&Middle East","Africa_S": "Africa&Middle East",
    "South Africa": "Africa&Middle East","South America_N": "South America",
    "South America_S": "South America","Brazil": "South America","Argentina": "South America",
    "European FTA": "Europe","Russia": "Europe","EU-12": "Europe","Europe_Eastern": "Europe",
    "Europe_Non_EU": "Europe","Central Asia": "Asia","Middle East": "Africa&Middle East",
    "South Asia": "Asia","Japan": "Asia","South Korea": "Asia","India": "Asia","Indonesia": "Asia",
    "China": "Asia","Unknown": "Other",
}

# 注意：这里沿用你给的新顺序，多了一个 Africa（目前没有映射到它的区域也没关系）
FIXED_CONTINENT_ORDER = ["Asia","Africa","Africa&Middle East","North America","South America","Oceania","Europe","Other"]

CONTINENT_BASE_COLORS = {
    "Asia": "#D8524B","Africa&Middle East": "#F39C12","North America": "#8E44AD",
    "South America": "#1A9850","Oceania": "#9E9E9E","Europe": "#4575B4","Other": "#BDBDBD",
}

# 中国专用颜色（深红）
CHINA_COLOR = "#C00000"

# ===============================
# 同一大洲不同地区用不同深浅（目前只在需要按地区配色时会用到）
# ===============================
def _mix_with_white(rgb, amount):
    rgb = np.array(mcolors.to_rgb(rgb))
    return tuple(np.clip(rgb*(1-amount) + amount*np.ones(3), 0, 1))

def build_region_colors(regions):
    """
    regions: 一个地区名称列表（比如 ['EU-15','Canada','USA', ...]）
    返回: 每个地区对应的颜色列表（顺序与 regions 一致）
    """
    cont_counts = defaultdict(int)
    for r in regions:
        cont_counts[REGION_TO_CONTINENT.get(r, "Other")] += 1

    cont_shades = {}
    for cont, cnt in cont_counts.items():
        base = CONTINENT_BASE_COLORS.get(cont, CONTINENT_BASE_COLORS["Other"])
        if cnt == 1:
            cont_shades[cont] = [base]
        else:
            amounts = np.linspace(0.10, 0.55, cnt)
            cont_shades[cont] = [_mix_with_white(base, a) for a in amounts]

    taken = {c: 0 for c in cont_counts}
    colors = []
    for r in regions:
        cont = REGION_TO_CONTINENT.get(r, "Other")
        idx = taken[cont]
        colors.append(cont_shades[cont][idx])
        taken[cont] += 1
    return colors

# ===============================
# 读取 RegionMapping_filter.csv
# ===============================
REGION_MAPPING_PATH = r'DataOutput/Used_output/RegionMapping_filter.csv'

region_code_to_name = {}

if os.path.exists(REGION_MAPPING_PATH):
    mapping_df = pd.read_csv(REGION_MAPPING_PATH, dtype=str)
    mapping_df.columns = [c.strip() for c in mapping_df.columns]

    # 自动识别“代码列”和“名称列”
    code_col = None
    name_col = None

    for col in mapping_df.columns:
        cl = col.lower()
        if 'code' in cl or cl in ['region', 'region_code', 'regionid', 'region_id']:
            code_col = col
            break

    for col in mapping_df.columns:
        cl = col.lower()
        if 'name' in cl or 'region_name' in cl:
            name_col = col
            break

    if code_col is None and len(mapping_df.columns) >= 1:
        code_col = mapping_df.columns[0]
    if name_col is None and len(mapping_df.columns) >= 2:
        name_col = mapping_df.columns[1]

    mapping_df[code_col] = mapping_df[code_col].astype(str).str.strip()
    mapping_df[name_col] = mapping_df[name_col].astype(str).str.strip()
    mapping_df[code_col] = mapping_df[code_col].str.replace(r'\.0$', '', regex=True)

    region_code_to_name = (
        mapping_df[[code_col, name_col]]
        .dropna()
        .set_index(code_col)[name_col]
        .to_dict()
    )

    print("RegionMapping_filter.csv 已读取：")
    print(f"  代码列: {code_col}, 名称列: {name_col}")
    print(mapping_df.head())
else:
    print("⚠ 未找到 RegionMapping_filter.csv，区域名称将使用代码本身。")

# ===============================
# 查找氢能数据文件并筛选情景
# ===============================
input_dir = 'DataOutput/Used_output/'
all_files = sorted(glob.glob(os.path.join(input_dir, '*_withPotential.csv')))

hydrogen_files_all = [
    f for f in all_files
    if 'hydrogen' in f.lower() or 'h2' in os.path.basename(f).lower()
]

print(f"找到 {len(hydrogen_files_all)} 个氢能潜力文件(原始):")
for f in hydrogen_files_all:
    print("  -", os.path.basename(f))

keep_years = ['2030', '2040', '2050', '2100']

files_2030 = [f for f in hydrogen_files_all if '2030' in os.path.basename(f)]
files_2050 = [f for f in hydrogen_files_all if '2050' in os.path.basename(f)]
files_noyear = [
    f for f in hydrogen_files_all
    if all(y not in os.path.basename(f) for y in keep_years)
]

hydrogen_files = sorted(files_2030) + sorted(files_noyear) + sorted(files_2050)
hydrogen_files = hydrogen_files[:3]  # 最多三个：2030 → 无年份 → 2050

print("\n用于绘图的情景文件：")
for f in hydrogen_files:
    print("  -", os.path.basename(f))

# ===============================
# 辅助函数：给定文件路径，读入并返回处理好的 tmp
# （方便主图和前1000Mt图重复使用）
# ===============================
def load_and_prepare_tmp(file_path):
    df = pd.read_csv(file_path)

    # 成本列
    cost_col = None
    for col in df.columns:
        if ('hydrogen' in col.lower() and 'cost' in col.lower()
            and ('kg' in col.lower() or 'ton' in col.lower())):
            cost_col = col
            break
    if cost_col is None:
        for col in ['levelized_hydrogen_cost_per_kg','hydrogen_cost_per_kg',
                    'levelized_hydrogen_cost_per_ton','hydrogen_cost_per_ton',
                    'levelized_cost_per_kg','cost_per_kg']:
            if col in df.columns:
                cost_col = col
                break

    # 潜力列
    potential_col = None
    conversion = 1.0
    if 'production_potential_Mt' in df.columns:
        potential_col = 'production_potential_Mt'
        conversion = 1.0
    elif 'production_potential_ton' in df.columns:
        potential_col = 'production_potential_ton'
        conversion = 1e-6
    elif 'production_potential_kg' in df.columns:
        potential_col = 'production_potential_kg'
        conversion = 1e-9
    elif 'production_potential' in df.columns:
        potential_col = 'production_potential'
        conversion = 1e-9  # 假设 kg

    # Region 列
    region_col = None
    for col in df.columns:
        if col.lower() in ['region', 'region_code', 'regionid', 'region_id']:
            region_col = col
            break
    if region_col is None:
        for col in df.columns:
            if 'region' in col.lower():
                region_col = col
                break

    if cost_col is None or potential_col is None:
        return None, cost_col  # 用 cost_col 是否为 None 来标记失败

    tmp = df.copy()
    tmp['cost'] = pd.to_numeric(tmp[cost_col], errors='coerce')
    tmp['potential_Mt'] = pd.to_numeric(tmp[potential_col], errors='coerce') * conversion

    if region_col is not None:
        tmp['RegionCode'] = tmp[region_col].fillna('Unknown').astype(str).str.strip()
    else:
        tmp['RegionCode'] = 'Unknown'

    tmp['RegionCode'] = tmp['RegionCode'].str.replace(r'\.0$', '', regex=True)

    if region_code_to_name:
        tmp['RegionName'] = tmp['RegionCode'].map(region_code_to_name).fillna(tmp['RegionCode'])
    else:
        tmp['RegionName'] = tmp['RegionCode']

    tmp['Continent'] = tmp['RegionName'].map(REGION_TO_CONTINENT).fillna('Other')

    tmp = tmp[
        np.isfinite(tmp['cost']) &
        np.isfinite(tmp['potential_Mt']) &
        (tmp['potential_Mt'] > 0)
    ]

    if len(tmp) == 0:
        return None, cost_col

    tmp = tmp.sort_values('cost')
    tmp['cumulative_Mt'] = tmp['potential_Mt'].cumsum()

    return tmp, cost_col

# ===============================
# 1) 原来的主图：全范围、按大洲填色 + 中国深红
#    同时记录每个情景的 y 轴范围，供前1000Mt图复用
# ===============================
main_ylim_dict = {}  # key: file_path, value: (ymin, ymax)

if len(hydrogen_files) == 0:
    print("\n未找到符合条件的氢能数据文件！")
else:
    n_plots = len(hydrogen_files)
    fig, axes = plt.subplots(n_plots, 1, figsize=(9, 12), sharex=False)
    if n_plots == 1:
        axes = [axes]

    for idx, file_path in enumerate(hydrogen_files):
        ax = axes[idx]
        file_name = os.path.basename(file_path).replace('combined_', '').replace('_withPotential.csv', '')

        print(f"\n处理文件（主图）: {file_name}")
        try:
            tmp, cost_col = load_and_prepare_tmp(file_path)
        except Exception as e:
            ax.text(0.5, 0.5, f'读取失败:\n{e}', ha='center', va='center', fontsize=14)
            ax.set_title(file_name, fontsize=11)
            continue

        if tmp is None or cost_col is None:
            ax.text(0.5, 0.5, '缺少成本或潜力列/无有效数据', ha='center', va='center', fontsize=14)
            ax.set_title(file_name, fontsize=11)
            continue

        min_cost = tmp['cost'].min()
        max_cost = tmp['cost'].max()
        total_potential = tmp['cumulative_Mt'].iloc[-1]

        print(f"  有效点数: {len(tmp)}, 成本范围: {min_cost:.2f}-{max_cost:.2f} $/kg, 总潜力: {total_potential:.1f} Mt")
        print("  按大洲统计潜力 (Mt)：")
        print(tmp.groupby('Continent')['potential_Mt'].sum())

        # 分段 x
        tmp['x_right'] = tmp['cumulative_Mt']
        tmp['x_left'] = tmp['cumulative_Mt'].shift(fill_value=0.0)
        tmp['width'] = tmp['x_right'] - tmp['x_left']
        tmp['x_center'] = tmp['x_left'] + tmp['width'] / 2.0

        # 按大洲画面积条 + 中国深红
        for cont in FIXED_CONTINENT_ORDER:
            sub = tmp[tmp['Continent'] == cont]
            if sub.empty:
                continue

            sub_china = sub[sub['RegionName'] == 'China']
            sub_normal = sub[sub['RegionName'] != 'China']

            base_color = CONTINENT_BASE_COLORS.get(cont, CONTINENT_BASE_COLORS["Other"])

            if not sub_normal.empty:
                ax.bar(
                    sub_normal['x_center'],
                    sub_normal['cost'],
                    width=sub_normal['width'],
                    bottom=0.0,
                    color=base_color,
                    edgecolor='none',
                    linewidth=0,
                    alpha=0.7,
                    align='center'
                )

            if not sub_china.empty:
                ax.bar(
                    sub_china['x_center'],
                    sub_china['cost'],
                    width=sub_china['width'],
                    bottom=0.0,
                    color=CHINA_COLOR,
                    edgecolor='none',
                    linewidth=0,
                    alpha=0.9,
                    align='center'
                )

        ax.plot(tmp['cumulative_Mt'], tmp['cost'], color='black', linewidth=1.5)

        ax.set_title(file_name, fontsize=14, pad=6)
        if idx == n_plots - 1:
            ax.set_xlabel('累积氢气生产潜力 (Mt)', fontsize=14)
        else:
            ax.set_xlabel('')

        if 'kg' in cost_col.lower():
            ax.set_ylabel('氢气平准化成本 ($/kg)', fontsize=14)
        else:
            ax.set_ylabel('氢气平准化成本 ($/ton)', fontsize=14)

        ax.tick_params(labelsize=14)
        ax.set_facecolor('white')

        info_text = f'可开发潜力: {total_potential:.1f} Mt\n成本: ${min_cost:.2f}-{max_cost:.2f}'
        ax.text(
            0.95, 0.95, info_text,
            transform=ax.transAxes,
            fontsize=14,
            ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.2',
                      facecolor='white', alpha=0.8,
                      edgecolor='gray', linewidth=0.8)
        )

        # 记录主图该情景的 y 轴范围
        main_ylim_dict[file_path] = ax.get_ylim()

    plt.tight_layout(rect=[0.03, 0.03, 0.97, 0.97])
    os.makedirs('04 Potenial Mapping', exist_ok=True)
    plt.savefig('04 Potenial Mapping/hydrogen_supply_curves_by_region.png', dpi=300, bbox_inches='tight')
    plt.savefig('04 Potenial Mapping/hydrogen_supply_curves_by_region.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    # 单独画大洲 + 中国 的图例
    legend_fig, legend_ax = plt.subplots(figsize=(7, 1.5))
    patches = []
    labels = []
    for cont in FIXED_CONTINENT_ORDER:
        if cont not in CONTINENT_BASE_COLORS:
            continue
        color = CONTINENT_BASE_COLORS.get(cont, CONTINENT_BASE_COLORS["Other"])
        label = CONTINENT_ZH.get(cont, cont)
        patches.append(mpatches.Patch(color=color))
        labels.append(label)
    patches.append(mpatches.Patch(color=CHINA_COLOR))
    labels.append("中国（单独标注）")

    legend_ax.legend(
        handles=patches,
        labels=labels,
        loc='center',
        ncol=len(patches),
        frameon=False,
        fontsize=10
    )
    legend_ax.axis('off')
    legend_fig.tight_layout()
    legend_fig.savefig('04 Potenial Mapping/hydrogen_supply_continent_legend.png', dpi=300, bbox_inches='tight')
    legend_fig.savefig('04 Potenial Mapping/hydrogen_supply_continent_legend.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n" + "="*70)
    print("主图（全范围，按大洲 + 中国深红）已保存：")
    print("  - hydrogen_supply_curves_by_region.png")
    print("  - hydrogen_supply_curves_by_region.pdf")
    print("大洲 + 中国图例：")
    print("  - hydrogen_supply_continent_legend.png")
    print("  - hydrogen_supply_continent_legend.pdf")
    print("="*70)

    # ===============================
    # 2) 前 1000 Mt 的单独绘图：按大洲配色 + 中国单独深红
    #    且 y 轴与对应主图保持一致
    #    同时收集出现过的大洲，用于单独画 top1000Mt 的大洲 legend
    # ===============================
    fig2, axes2 = plt.subplots(n_plots, 1, figsize=(6, 12), sharex=False)
    if n_plots == 1:
        axes2 = [axes2]

    continent_legend_order = []  # 保持大洲顺序
    continents_used = set()
    china_used = False

    for idx, file_path in enumerate(hydrogen_files):
        ax2 = axes2[idx]
        file_name = os.path.basename(file_path).replace('combined_', '').replace('_withPotential.csv', '')

        print(f"\n处理文件（前1000Mt）：{file_name}")
        try:
            tmp, cost_col = load_and_prepare_tmp(file_path)
        except Exception as e:
            ax2.text(0.5, 0.5, f'读取失败:\n{e}', ha='center', va='center', fontsize=14)
            ax2.set_title(file_name + " (前1000 Mt)", fontsize=14)
            continue

        if tmp is None or cost_col is None:
            ax2.text(0.5, 0.5, '缺少成本或潜力列/无有效数据', ha='center', va='center', fontsize=14)
            ax2.set_title(file_name + " (前1000 Mt)", fontsize=14)
            continue

        # 只取累计潜力 <= 1000 Mt 的部分
        tmp_sub = tmp[tmp['cumulative_Mt'] <= 1000].copy()
        if len(tmp_sub) == 0:
            ax2.text(0.5, 0.5, '前1000 Mt 区间无数据', ha='center', va='center', fontsize=14)
            ax2.set_title(file_name + " (前1000 Mt)", fontsize=14)
            continue

        # 分段 x
        tmp_sub['x_right'] = tmp_sub['cumulative_Mt']
        tmp_sub['x_left'] = tmp_sub['cumulative_Mt'].shift(fill_value=0.0)
        tmp_sub['width'] = tmp_sub['x_right'] - tmp_sub['x_left']
        tmp_sub['x_center'] = tmp_sub['x_left'] + tmp_sub['width'] / 2.0

        # 记录本情景中出现过的大洲 & 是否出现过中国
        sub_continents = tmp_sub['Continent'].unique()
        for c in sub_continents:
            if c not in continents_used and c in CONTINENT_BASE_COLORS:
                continents_used.add(c)
                continent_legend_order.append(c)
        if (tmp_sub['RegionName'] == 'China').any():
            china_used = True

        # 按大洲画面积条 + 中国深红（与主图用同一套逻辑）
        for cont in FIXED_CONTINENT_ORDER:
            sub = tmp_sub[tmp_sub['Continent'] == cont]
            if sub.empty:
                continue

            sub_china = sub[sub['RegionName'] == 'China']
            sub_normal = sub[sub['RegionName'] != 'China']

            base_color = CONTINENT_BASE_COLORS.get(cont, CONTINENT_BASE_COLORS["Other"])

            if not sub_normal.empty:
                ax2.bar(
                    sub_normal['x_center'],
                    sub_normal['cost'],
                    width=sub_normal['width'],
                    bottom=0.0,
                    color=base_color,
                    edgecolor='none',
                    linewidth=0,
                    alpha=0.9,
                    align='center'
                )

            if not sub_china.empty:
                ax2.bar(
                    sub_china['x_center'],
                    sub_china['cost'],
                    width=sub_china['width'],
                    bottom=0.0,
                    color=CHINA_COLOR,
                    edgecolor='none',
                    linewidth=0,
                    alpha=0.9,
                    align='center'
                )

        # 前1000 Mt 供应曲线
        ax2.plot(tmp_sub['cumulative_Mt'], tmp_sub['cost'], color='black', linewidth=1.5)

        min_cost_sub = tmp_sub['cost'].min()
        max_cost_sub = tmp_sub['cost'].max()
        total_potential_sub = tmp_sub['cumulative_Mt'].iloc[-1]

        ax2.set_title(file_name + "（前1000 Mt）", fontsize=14, pad=6)
        if idx == n_plots - 1:
            ax2.set_xlabel('累积氢气生产潜力 (Mt)', fontsize=14)
        else:
            ax2.set_xlabel('')

        if 'kg' in cost_col.lower():
            ax2.set_ylabel('氢气平准化成本 ($/kg)', fontsize=14)
        else:
            ax2.set_ylabel('氢气平准化成本 ($/ton)', fontsize=14)

        ax2.tick_params(labelsize=14)
        ax2.set_facecolor('white')
        ax2.set_xlim(left=0, right=max(1000, total_potential_sub * 1.01))

        # 让前1000Mt图的 y 轴与对应主图保持一致
        if file_path in main_ylim_dict:
            ax2.set_ylim(main_ylim_dict[file_path])

        info_text2 = f'成本: ${min_cost_sub:.2f}-{max_cost_sub:.2f}'
        ax2.text(
            0.95, 0.95, info_text2,
            transform=ax2.transAxes,
            fontsize=14,
            ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.2',
                      facecolor='white', alpha=0.8,
                      edgecolor='gray', linewidth=0.8)
        )

    plt.tight_layout(rect=[0.03, 0.03, 0.97, 0.97])
    plt.savefig('04 Potenial Mapping/hydrogen_supply_curves_top1000Mt_by_region.png', dpi=300, bbox_inches='tight')
    plt.savefig('04 Potenial Mapping/hydrogen_supply_curves_top1000Mt_by_region.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n" + "="*70)
    print("前1000 Mt、按大洲配色的图已保存：")
    print("  - hydrogen_supply_curves_top1000Mt_by_region.png")
    print("  - hydrogen_supply_curves_top1000Mt_by_region.pdf")
    print("="*70)

    # ===============================
    # 3) 单独绘制前1000Mt的大洲 legend（只包含实际出现的大洲 + 中国）
    # ===============================
    if continent_legend_order or china_used:
        n_items = len(continent_legend_order) + (1 if china_used else 0)
        fig_width = max(6, 0.7 * n_items)
        region_legend_fig, region_legend_ax = plt.subplots(figsize=(fig_width, 1.8))

        legend_patches = []
        legend_labels = []

        for cont in continent_legend_order:
            color = CONTINENT_BASE_COLORS.get(cont, CONTINENT_BASE_COLORS["Other"])
            label = CONTINENT_ZH.get(cont, cont)
            legend_patches.append(mpatches.Patch(color=color))
            legend_labels.append(label)

        if china_used:
            legend_patches.append(mpatches.Patch(color=CHINA_COLOR))
            legend_labels.append("中国（单独标注）")

        region_legend_ax.legend(
            handles=legend_patches,
            labels=legend_labels,
            loc='center',
            ncol=len(legend_patches),
            frameon=False,
            fontsize=9
        )
        region_legend_ax.axis('off')
        region_legend_fig.tight_layout()
        region_legend_fig.savefig('04 Potenial Mapping/hydrogen_supply_continent_top1000Mt_legend.png', dpi=300, bbox_inches='tight')
        region_legend_fig.savefig('04 Potenial Mapping/hydrogen_supply_continent_top1000Mt_legend.pdf', dpi=300, bbox_inches='tight')
        plt.show()

        print("\n前1000Mt 大洲图例已保存：")
        print("  - hydrogen_supply_continent_top1000Mt_legend.png")
        print("  - hydrogen_supply_continent_top1000Mt_legend.pdf")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import matplotlib.patches as mpatches
from collections import defaultdict
import matplotlib.colors as mcolors

# ===============================
# Matplotlib 风格
# ===============================
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 14
plt.rcParams['axes.linewidth'] = 1.4
plt.rcParams['lines.linewidth'] = 2.0

# ===============================
# 英文 → 中文映射
# ===============================
CONTINENT_ZH = {
    "Asia": "亚洲",
    "Africa&Middle East": "非洲与中东",
    "North America": "北美",
    "South America": "南美",
    "Oceania": "大洋洲",
    "Europe": "欧洲",
    "Other": "其他",
}

REGION_NAME_ZH = {
    "EU-15": "欧盟-15","Canada": "加拿大","USA": "美国","Central America": "中美洲",
    "Mexico": "墨西哥","Southeast Asia": "东南亚","Australia_NZ": "澳大利亚与新西兰",
    "#N/A": "无","Africa_N": "北非","Africa_W": "西非","Africa_E": "东非","Africa_S": "南部非洲",
    "South Africa": "南非","South America_N": "南美洲北部","South America_S": "南美洲南部",
    "Brazil": "巴西","Argentina": "阿根廷","European FTA": "欧洲自由贸易联盟","Russia": "俄罗斯",
    "EU-12": "欧盟-12","Europe_Eastern": "东欧","Europe_Non_EU": "欧洲（非欧盟）",
    "Central Asia": "中亚","Middle East": "中东","South Asia": "南亚","Japan": "日本",
    "South Korea": "韩国","India": "印度","Indonesia": "印度尼西亚","China": "中国","Unknown": "未知",
}

# ===============================
# 区域→大洲映射与顺序
# ===============================
REGION_TO_CONTINENT = {
    "EU-15": "Europe","Canada": "North America","USA": "North America",
    "Central America": "South America","Mexico": "North America",
    "Southeast Asia": "Asia","Australia_NZ": "Oceania","#N/A": "Other",
    "Africa_N": "Africa&Middle East","Africa_W": "Africa&Middle East",
    "Africa_E": "Africa&Middle East","Africa_S": "Africa&Middle East",
    "South Africa": "Africa&Middle East","South America_N": "South America",
    "South America_S": "South America","Brazil": "South America","Argentina": "South America",
    "European FTA": "Europe","Russia": "Europe","EU-12": "Europe","Europe_Eastern": "Europe",
    "Europe_Non_EU": "Europe","Central Asia": "Asia","Middle East": "Africa&Middle East",
    "South Asia": "Asia","Japan": "Asia","South Korea": "Asia","India": "Asia","Indonesia": "Asia",
    "China": "Asia","Unknown": "Other",
}

FIXED_CONTINENT_ORDER = ["Asia","Africa","Africa&Middle East","North America","South America","Oceania","Europe","Other"]

CONTINENT_BASE_COLORS = {
    "Asia": "#D8524B","Africa&Middle East": "#F39C12","North America": "#8E44AD",
    "South America": "#1A9850","Oceania": "#9E9E9E","Europe": "#4575B4","Other": "#BDBDBD",
}

# 中国专用颜色（深红）
CHINA_COLOR = "#C00000"

# ===============================
# 同一大洲不同地区用不同深浅（目前没用到，但保留函数）
# ===============================
def _mix_with_white(rgb, amount):
    rgb = np.array(mcolors.to_rgb(rgb))
    return tuple(np.clip(rgb*(1-amount) + amount*np.ones(3), 0, 1))

def build_region_colors(regions):
    cont_counts = defaultdict(int)
    for r in regions:
        cont_counts[REGION_TO_CONTINENT.get(r, "Other")] += 1

    cont_shades = {}
    for cont, cnt in cont_counts.items():
        base = CONTINENT_BASE_COLORS.get(cont, CONTINENT_BASE_COLORS["Other"])
        if cnt == 1:
            cont_shades[cont] = [base]
        else:
            amounts = np.linspace(0.10, 0.55, cnt)
            cont_shades[cont] = [_mix_with_white(base, a) for a in amounts]

    taken = {c: 0 for c in cont_counts}
    colors = []
    for r in regions:
        cont = REGION_TO_CONTINENT.get(r, "Other")
        idx = taken[cont]
        colors.append(cont_shades[cont][idx])
        taken[cont] += 1
    return colors

# ===============================
# 读取 RegionMapping_filter.csv
# ===============================
REGION_MAPPING_PATH = r'DataOutput/Used_output/RegionMapping_filter.csv'

region_code_to_name = {}

if os.path.exists(REGION_MAPPING_PATH):
    mapping_df = pd.read_csv(REGION_MAPPING_PATH, dtype=str)
    mapping_df.columns = [c.strip() for c in mapping_df.columns]

    code_col = None
    name_col = None

    for col in mapping_df.columns:
        cl = col.lower()
        if 'code' in cl or cl in ['region', 'region_code', 'regionid', 'region_id']:
            code_col = col
            break

    for col in mapping_df.columns:
        cl = col.lower()
        if 'name' in cl or 'region_name' in cl:
            name_col = col
            break

    if code_col is None and len(mapping_df.columns) >= 1:
        code_col = mapping_df.columns[0]
    if name_col is None and len(mapping_df.columns) >= 2:
        name_col = mapping_df.columns[1]

    mapping_df[code_col] = mapping_df[code_col].astype(str).str.strip()
    mapping_df[name_col] = mapping_df[name_col].astype(str).str.strip()
    mapping_df[code_col] = mapping_df[code_col].str.replace(r'\.0$', '', regex=True)

    region_code_to_name = (
        mapping_df[[code_col, name_col]]
        .dropna()
        .set_index(code_col)[name_col]
        .to_dict()
    )

    print("RegionMapping_filter.csv 已读取：")
    print(f"  代码列: {code_col}, 名称列: {name_col}")
    print(mapping_df.head())
else:
    print("⚠ 未找到 RegionMapping_filter.csv，区域名称将使用代码本身。")

# ===============================
# 通用读入函数：根据燃料类型识别成本列
# ===============================
def load_and_prepare_tmp_fuel(file_path, fuel_type):
    """
    fuel_type: 'ammonia' 或 'methanol'
    """
    df = pd.read_csv(file_path)

    ft = fuel_type.lower()
    if ft == 'ammonia':
        fuel_keywords = ['ammonia', 'nh3']
        fallback_cost_cols = [
            'levelized_ammonia_cost_per_kg','ammonia_cost_per_kg',
            'levelized_nh3_cost_per_kg','nh3_cost_per_kg',
            'ammonia_cost_per_ton','nh3_cost_per_ton',
            'levelized_cost_per_kg','cost_per_kg'
        ]
    elif ft == 'methanol':
        fuel_keywords = ['methanol', 'meoh']
        fallback_cost_cols = [
            'levelized_methanol_cost_per_kg','methanol_cost_per_kg',
            'levelized_meoh_cost_per_kg','meoh_cost_per_kg',
            'methanol_cost_per_ton','meoh_cost_per_ton',
            'levelized_cost_per_kg','cost_per_kg'
        ]
    else:
        fuel_keywords = []
        fallback_cost_cols = ['levelized_cost_per_kg','cost_per_kg']

    # 成本列识别
    cost_col = None
    for col in df.columns:
        cl = col.lower()
        if 'cost' in cl and ('kg' in cl or 'ton' in cl) and any(k in cl for k in fuel_keywords):
            cost_col = col
            break
    if cost_col is None:
        for col in fallback_cost_cols:
            if col in df.columns:
                cost_col = col
                break

    # 潜力列（假定命名不带燃料前缀，和氢气一样）
    potential_col = None
    conversion = 1.0
    if 'production_potential_Mt' in df.columns:
        potential_col = 'production_potential_Mt'
        conversion = 1.0
    elif 'production_potential_ton' in df.columns:
        potential_col = 'production_potential_ton'
        conversion = 1e-6
    elif 'production_potential_kg' in df.columns:
        potential_col = 'production_potential_kg'
        conversion = 1e-9
    elif 'production_potential' in df.columns:
        potential_col = 'production_potential'
        conversion = 1e-9  # 假设 kg

    # Region 列
    region_col = None
    for col in df.columns:
        if col.lower() in ['region', 'region_code', 'regionid', 'region_id']:
            region_col = col
            break
    if region_col is None:
        for col in df.columns:
            if 'region' in col.lower():
                region_col = col
                break

    if cost_col is None or potential_col is None:
        return None, cost_col

    tmp = df.copy()
    tmp['cost'] = pd.to_numeric(tmp[cost_col], errors='coerce')
    tmp['potential_Mt'] = pd.to_numeric(tmp[potential_col], errors='coerce') * conversion

    if region_col is not None:
        tmp['RegionCode'] = tmp[region_col].fillna('Unknown').astype(str).str.strip()
    else:
        tmp['RegionCode'] = 'Unknown'

    tmp['RegionCode'] = tmp['RegionCode'].str.replace(r'\.0$', '', regex=True)

    if region_code_to_name:
        tmp['RegionName'] = tmp['RegionCode'].map(region_code_to_name).fillna(tmp['RegionCode'])
    else:
        tmp['RegionName'] = tmp['RegionCode']

    tmp['Continent'] = tmp['RegionName'].map(REGION_TO_CONTINENT).fillna('Other')

    tmp = tmp[
        np.isfinite(tmp['cost']) &
        np.isfinite(tmp['potential_Mt']) &
        (tmp['potential_Mt'] > 0)
    ]

    if len(tmp) == 0:
        return None, cost_col

    tmp = tmp.sort_values('cost')
    tmp['cumulative_Mt'] = tmp['potential_Mt'].cumsum()

    return tmp, cost_col

# ===============================
# 通用绘图函数（供 Ammonia / Methanol 复用）
# ===============================
def plot_supply_curves_for_fuel(fuel_type,
                                file_keywords,
                                output_prefix,
                                y_label_kg,
                                y_label_ton,
                                top_Mt):
    """
    fuel_type: 'ammonia' or 'methanol'
    file_keywords: 在文件名中用来筛选该燃料的关键词列表（小写）
    output_prefix: 输出文件名前缀，如 'ammonia' 或 'methanol'
    top_Mt: 只绘制前 top_Mt Mt 的子图（例如 1000, 3000）
    """
    input_dir = 'DataOutput/Used_output/'
    all_files = sorted(glob.glob(os.path.join(input_dir, '*_withPotential.csv')))

    # 筛选对应燃料的文件
    fuel_files_all = [
        f for f in all_files
        if any(k in os.path.basename(f).lower() for k in file_keywords)
    ]

    print(f"\n==== {fuel_type.upper()}：找到 {len(fuel_files_all)} 个潜力文件(原始) ====")
    for f in fuel_files_all:
        print("  -", os.path.basename(f))

    keep_years = ['2030', '2040', '2050', '2100']

    files_2030 = [f for f in fuel_files_all if '2030' in os.path.basename(f)]
    files_2050 = [f for f in fuel_files_all if '2050' in os.path.basename(f)]
    files_noyear = [
        f for f in fuel_files_all
        if all(y not in os.path.basename(f) for y in keep_years)
    ]

    fuel_files = sorted(files_2030) + sorted(files_noyear) + sorted(files_2050)
    fuel_files = fuel_files[:3]

    print(f"\n用于绘制 {fuel_type} 供应曲线的情景文件：")
    for f in fuel_files:
        print("  -", os.path.basename(f))

    if len(fuel_files) == 0:
        print(f"\n⚠ 未找到符合条件的 {fuel_type} 数据文件！")
        return

    # -------- 1) 主图：全范围、按大洲填色 + 中国深红 --------
    main_ylim_dict = {}

    n_plots = len(fuel_files)
    fig, axes = plt.subplots(n_plots, 1, figsize=(9, 12), sharex=False)
    if n_plots == 1:
        axes = [axes]

    for idx, file_path in enumerate(fuel_files):
        ax = axes[idx]
        file_name = os.path.basename(file_path).replace('combined_', '').replace('_withPotential.csv', '')

        print(f"\n处理文件（主图, {fuel_type}）: {file_name}")
        try:
            tmp, cost_col = load_and_prepare_tmp_fuel(file_path, fuel_type)
        except Exception as e:
            ax.text(0.5, 0.5, f'读取失败:\n{e}', ha='center', va='center', fontsize=14)
            ax.set_title(file_name, fontsize=14)
            continue

        if tmp is None or cost_col is None:
            ax.text(0.5, 0.5, '缺少成本或潜力列/无有效数据', ha='center', va='center', fontsize=14)
            ax.set_title(file_name, fontsize=14)
            continue

        min_cost = tmp['cost'].min()
        max_cost = tmp['cost'].max()
        total_potential = tmp['cumulative_Mt'].iloc[-1]

        print(f"  有效点数: {len(tmp)}, 成本范围: {min_cost:.2f}-{max_cost:.2f}, 总潜力: {total_potential:.1f} Mt")
        print("  按大洲统计潜力 (Mt)：")
        print(tmp.groupby('Continent')['potential_Mt'].sum())

        tmp['x_right'] = tmp['cumulative_Mt']
        tmp['x_left'] = tmp['cumulative_Mt'].shift(fill_value=0.0)
        tmp['width'] = tmp['x_right'] - tmp['x_left']
        tmp['x_center'] = tmp['x_left'] + tmp['width'] / 2.0

        for cont in FIXED_CONTINENT_ORDER:
            sub = tmp[tmp['Continent'] == cont]
            if sub.empty:
                continue

            sub_china = sub[sub['RegionName'] == 'China']
            sub_normal = sub[sub['RegionName'] != 'China']

            base_color = CONTINENT_BASE_COLORS.get(cont, CONTINENT_BASE_COLORS["Other"])

            if not sub_normal.empty:
                ax.bar(
                    sub_normal['x_center'],
                    sub_normal['cost'],
                    width=sub_normal['width'],
                    bottom=0.0,
                    color=base_color,
                    edgecolor='none',
                    linewidth=0,
                    alpha=0.7,
                    align='center'
                )

            if not sub_china.empty:
                ax.bar(
                    sub_china['x_center'],
                    sub_china['cost'],
                    width=sub_china['width'],
                    bottom=0.0,
                    color=CHINA_COLOR,
                    edgecolor='none',
                    linewidth=0,
                    alpha=0.9,
                    align='center'
                )

        ax.plot(tmp['cumulative_Mt'], tmp['cost'], color='black', linewidth=1.5)

        ax.set_title(file_name, fontsize=14, pad=6)
        if idx == n_plots - 1:
            ax.set_xlabel(f'累积{fuel_type.capitalize()}生产潜力 (Mt)', fontsize=14)
        else:
            ax.set_xlabel('')

        if 'kg' in cost_col.lower():
            ax.set_ylabel(y_label_kg, fontsize=14)
        else:
            ax.set_ylabel(y_label_ton, fontsize=14)

        ax.tick_params(labelsize=14)
        ax.set_facecolor('white')

        info_text = f'经济可开发潜力: {total_potential:.1f} Mt\n成本: ${min_cost:.2f}-{max_cost:.2f}'
        ax.text(
            0.95, 0.95, info_text,
            transform=ax.transAxes,
            fontsize=14,
            ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.2',
                      facecolor='white', alpha=0.8,
                      edgecolor='gray', linewidth=0.8)
        )

        main_ylim_dict[file_path] = ax.get_ylim()

    plt.tight_layout(rect=[0.03, 0.03, 0.97, 0.97])
    os.makedirs('04 Potenial Mapping', exist_ok=True)
    plt.savefig(f'04 Potenial Mapping/{output_prefix}_supply_curves_by_region.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'04 Potenial Mapping/{output_prefix}_supply_curves_by_region.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    # 大洲 + 中国 图例
    legend_fig, legend_ax = plt.subplots(figsize=(7, 1.5))
    patches = []
    labels = []
    for cont in FIXED_CONTINENT_ORDER:
        if cont not in CONTINENT_BASE_COLORS:
            continue
        color = CONTINENT_BASE_COLORS.get(cont, CONTINENT_BASE_COLORS["Other"])
        label = CONTINENT_ZH.get(cont, cont)
        patches.append(mpatches.Patch(color=color))
        labels.append(label)
    patches.append(mpatches.Patch(color=CHINA_COLOR))
    labels.append("中国（单独标注）")

    legend_ax.legend(
        handles=patches,
        labels=labels,
        loc='center',
        ncol=len(patches),
        frameon=False,
        fontsize=10
    )
    legend_ax.axis('off')
    legend_fig.tight_layout()
    legend_fig.savefig(f'04 Potenial Mapping/{output_prefix}_supply_continent_legend.png', dpi=300, bbox_inches='tight')
    legend_fig.savefig(f'04 Potenial Mapping/{output_prefix}_supply_continent_legend.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n" + "="*70)
    print(f"{fuel_type} 主图（全范围，按大洲 + 中国深红）已保存：")
    print(f"  - {output_prefix}_supply_curves_by_region.png")
    print(f"  - {output_prefix}_supply_curves_by_region.pdf")
    print(f"{fuel_type} 大洲 + 中国图例：")
    print(f"  - {output_prefix}_supply_continent_legend.png")
    print(f"  - {output_prefix}_supply_continent_legend.pdf")
    print("="*70)

    # -------- 2) 前 top_Mt Mt：按大洲 + 中国，复用 y 轴 --------
    fig2, axes2 = plt.subplots(n_plots, 1, figsize=(5, 10), sharex=False)
    if n_plots == 1:
        axes2 = [axes2]

    continent_legend_order = []
    continents_used = set()
    china_used = False

    for idx, file_path in enumerate(fuel_files):
        ax2 = axes2[idx]
        file_name = os.path.basename(file_path).replace('combined_', '').replace('_withPotential.csv', '')

        print(f"\n处理文件（前{top_Mt}Mt, {fuel_type}）：{file_name}")
        try:
            tmp, cost_col = load_and_prepare_tmp_fuel(file_path, fuel_type)
        except Exception as e:
            ax2.text(0.5, 0.5, f'读取失败:\n{e}', ha='center', va='center', fontsize=14)
            ax2.set_title(file_name + f" (前{top_Mt} Mt)", fontsize=14)
            continue

        if tmp is None or cost_col is None:
            ax2.text(0.5, 0.5, '缺少成本或潜力列/无有效数据', ha='center', va='center', fontsize=14)
            ax2.set_title(file_name + f" (前{top_Mt} Mt)", fontsize=14)
            continue

        # 关键：这里用 top_Mt
        tmp_sub = tmp[tmp['cumulative_Mt'] <= top_Mt].copy()
        if len(tmp_sub) == 0:
            ax2.text(0.5, 0.5, f'前{top_Mt} Mt 区间无数据', ha='center', va='center', fontsize=14)
            ax2.set_title(file_name + f" (前{top_Mt} Mt)", fontsize=14)
            continue

        tmp_sub['x_right'] = tmp_sub['cumulative_Mt']
        tmp_sub['x_left'] = tmp_sub['cumulative_Mt'].shift(fill_value=0.0)
        tmp_sub['width'] = tmp_sub['x_right'] - tmp_sub['x_left']
        tmp_sub['x_center'] = tmp_sub['x_left'] + tmp_sub['width'] / 2.0

        sub_continents = tmp_sub['Continent'].unique()
        for c in sub_continents:
            if c not in continents_used and c in CONTINENT_BASE_COLORS:
                continents_used.add(c)
                continent_legend_order.append(c)
        if (tmp_sub['RegionName'] == 'China').any():
            china_used = True

        for cont in FIXED_CONTINENT_ORDER:
            sub = tmp_sub[tmp_sub['Continent'] == cont]
            if sub.empty:
                continue

            sub_china = sub[sub['RegionName'] == 'China']
            sub_normal = sub[sub['RegionName'] != 'China']

            base_color = CONTINENT_BASE_COLORS.get(cont, CONTINENT_BASE_COLORS["Other"])

            if not sub_normal.empty:
                ax2.bar(
                    sub_normal['x_center'],
                    sub_normal['cost'],
                    width=sub_normal['width'],
                    bottom=0.0,
                    color=base_color,
                    edgecolor='none',
                    linewidth=0,
                    alpha=0.9,
                    align='center'
                )

            if not sub_china.empty:
                ax2.bar(
                    sub_china['x_center'],
                    sub_china['cost'],
                    width=sub_china['width'],
                    bottom=0.0,
                    color=CHINA_COLOR,
                    edgecolor='none',
                    linewidth=0,
                    alpha=0.9,
                    align='center'
                )

        ax2.plot(tmp_sub['cumulative_Mt'], tmp_sub['cost'], color='black', linewidth=1.5)

        min_cost_sub = tmp_sub['cost'].min()
        max_cost_sub = tmp_sub['cost'].max()
        total_potential_sub = tmp_sub['cumulative_Mt'].iloc[-1]

        ax2.set_title(file_name + f"（前{top_Mt} Mt）", fontsize=14, pad=6)
        if idx == n_plots - 1:
            ax2.set_xlabel(f'累积{fuel_type.capitalize()}生产潜力 (Mt)', fontsize=14)
        else:
            ax2.set_xlabel('')

        if 'kg' in cost_col.lower():
            ax2.set_ylabel(y_label_kg, fontsize=14)
        else:
            ax2.set_ylabel(y_label_ton, fontsize=14)

        ax2.tick_params(labelsize=14)
        ax2.set_facecolor('white')
        ax2.set_xlim(left=0, right=max(top_Mt, total_potential_sub * 1.01))

        if file_path in main_ylim_dict:
            ax2.set_ylim(main_ylim_dict[file_path])

        info_text2 = f'成本: ${min_cost_sub:.2f}-{max_cost_sub:.2f}'
        ax2.text(
            0.95, 0.95, info_text2,
            transform=ax2.transAxes,
            fontsize=14,
            ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.2',
                      facecolor='white', alpha=0.8,
                      edgecolor='gray', linewidth=0.8)
        )

    plt.tight_layout(rect=[0.03, 0.03, 0.97, 0.97])
    plt.savefig(f'04 Potenial Mapping/{output_prefix}_supply_curves_top{top_Mt}Mt_by_region.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'04 Potenial Mapping/{output_prefix}_supply_curves_top{top_Mt}Mt_by_region.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n" + "="*70)
    print(f"{fuel_type} 前{top_Mt} Mt、按大洲配色的图已保存：")
    print(f"  - {output_prefix}_supply_curves_top{top_Mt}Mt_by_region.png")
    print(f"  - {output_prefix}_supply_curves_top{top_Mt}Mt_by_region.pdf")
    print("="*70)

    # -------- 3) 前 top_Mt Mt 大洲 legend（只包含实际出现的大洲 + 中国） --------
    if continent_legend_order or china_used:
        n_items = len(continent_legend_order) + (1 if china_used else 0)
        fig_width = max(6, 0.7 * n_items)
        legend_fig2, legend_ax2 = plt.subplots(figsize=(fig_width, 1.8))

        legend_patches = []
        legend_labels = []

        for cont in continent_legend_order:
            color = CONTINENT_BASE_COLORS.get(cont, CONTINENT_BASE_COLORS["Other"])
            label = CONTINENT_ZH.get(cont, cont)
            legend_patches.append(mpatches.Patch(color=color))
            legend_labels.append(label)

        if china_used:
            legend_patches.append(mpatches.Patch(color=CHINA_COLOR))
            legend_labels.append("中国（单独标注）")

        legend_ax2.legend(
            handles=legend_patches,
            labels=legend_labels,
            loc='center',
            ncol=len(legend_patches),
            frameon=False,
            fontsize=9
        )
        legend_ax2.axis('off')
        legend_fig2.tight_layout()
        legend_fig2.savefig(f'04 Potenial Mapping/{output_prefix}_supply_continent_top{top_Mt}Mt_legend.png', dpi=300, bbox_inches='tight')
        legend_fig2.savefig(f'04 Potenial Mapping/{output_prefix}_supply_continent_top{top_Mt}Mt_legend.pdf', dpi=300, bbox_inches='tight')
        plt.show()

        print(f"\n{fuel_type} 前{top_Mt}Mt 大洲图例已保存：")
        print(f"  - {output_prefix}_supply_continent_top{top_Mt}Mt_legend.png")
        print(f"  - {output_prefix}_supply_continent_top{top_Mt}Mt_legend.pdf")


# ===============================
# 实际调用：Ammonia & Methanol
# ===============================
# 氨：前 1000 Mt
plot_supply_curves_for_fuel(
    fuel_type='ammonia',
    file_keywords=['ammonia', 'nh3'],
    output_prefix='ammonia',
    y_label_kg='绿氨平准化成本 ($/kg)',
    y_label_ton='绿氨平准化成本 ($/ton)',
    top_Mt=1000
)

# 甲醇：前 1000 Mt
plot_supply_curves_for_fuel(
    fuel_type='methanol',
    file_keywords=['methanol', 'meoh'],
    output_prefix='methanol',
    y_label_kg='绿醇平准化成本 ($/kg)',
    y_label_ton='绿醇平准化成本 ($/ton)',
    top_Mt=1000
)


## 诊断：检查不同年份氢能潜力计算的问题

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# =========================
# 诊断潜力计算问题
# =========================

input_dir = 'DataOutput/Used_output/'
all_files = sorted(glob.glob(os.path.join(input_dir, '*_withPotential.csv')))
hydrogen_files = [f for f in all_files if 'hydrogen' in f.lower() or 'h2' in os.path.basename(f).lower()]

print("="*80)
print("氢能数据文件诊断报告")
print("="*80)

if len(hydrogen_files) == 0:
    print("\n未找到氢能数据文件！")
else:
    results = []
    
    for file_path in hydrogen_files:
        file_name = os.path.basename(file_path)
        print(f"\n{'='*80}")
        print(f"文件: {file_name}")
        print('='*80)
        
        try:
            df = pd.read_csv(file_path)
            print(f"总行数: {len(df)}")
            print(f"\n所有列名:")
            for i, col in enumerate(df.columns, 1):
                print(f"  {i}. {col}")
            
            # 查找潜力相关的列
            potential_cols = [col for col in df.columns if 'potential' in col.lower()]
            print(f"\n潜力相关列: {potential_cols}")
            
            # 查找成本相关的列
            cost_cols = [col for col in df.columns if 'cost' in col.lower() and 'hydrogen' in col.lower()]
            print(f"成本相关列: {cost_cols}")
            
            # 检查每个潜力列的统计信息
            for pot_col in potential_cols:
                print(f"\n列 '{pot_col}' 的统计信息:")
                valid_data = pd.to_numeric(df[pot_col], errors='coerce')
                valid_data = valid_data[np.isfinite(valid_data)]
                
                if len(valid_data) > 0:
                    print(f"  有效数据点: {len(valid_data)}/{len(df)}")
                    print(f"  最小值: {valid_data.min():.6e}")
                    print(f"  最大值: {valid_data.max():.6e}")
                    print(f"  均值: {valid_data.mean():.6e}")
                    print(f"  中位数: {valid_data.median():.6e}")
                    print(f"  总和: {valid_data.sum():.6e}")
                    print(f"  非零值数量: {(valid_data > 0).sum()}")
                    
                    # 推测单位
                    total = valid_data.sum()
                    if total < 1e6:
                        unit_guess = "Mt"
                        total_Mt = total
                    elif total < 1e9:
                        unit_guess = "ton"
                        total_Mt = total / 1e6
                    elif total < 1e12:
                        unit_guess = "kg"
                        total_Mt = total / 1e9
                    else:
                        unit_guess = "g"
                        total_Mt = total / 1e12
                    
                    print(f"  推测单位: {unit_guess}")
                    print(f"  转换为 Mt: {total_Mt:.2f} Mt")
                    
                    # 保存结果
                    results.append({
                        'file': file_name,
                        'column': pot_col,
                        'unit_guess': unit_guess,
                        'total_original': total,
                        'total_Mt': total_Mt,
                        'valid_points': len(valid_data),
                        'non_zero': (valid_data > 0).sum()
                    })
                else:
                    print(f"  无有效数据")
            
            # 如果有成本列，检查其统计信息
            for cost_col in cost_cols:
                print(f"\n列 '{cost_col}' 的统计信息:")
                valid_cost = pd.to_numeric(df[cost_col], errors='coerce')
                valid_cost = valid_cost[np.isfinite(valid_cost)]
                
                if len(valid_cost) > 0:
                    print(f"  有效数据点: {len(valid_cost)}/{len(df)}")
                    print(f"  最小值: ${valid_cost.min():.4f}")
                    print(f"  最大值: ${valid_cost.max():.4f}")
                    print(f"  均值: ${valid_cost.mean():.4f}")
                    
                    # 判断单位
                    avg_cost = valid_cost.mean()
                    if avg_cost < 10:
                        unit_guess = "$/kg"
                    else:
                        unit_guess = "$/ton"
                    print(f"  推测单位: {unit_guess}")
        
        except Exception as e:
            print(f"读取文件出错: {e}")
            import traceback
            traceback.print_exc()
    
    # 生成汇总表
    if results:
        print("\n" + "="*80)
        print("汇总表：所有文件的潜力对比")
        print("="*80)
        summary_df = pd.DataFrame(results)
        print(summary_df.to_string(index=False))
        
        print("\n" + "="*80)
        print("按文件分组的总潜力 (Mt)")
        print("="*80)
        file_totals = summary_df.groupby('file')['total_Mt'].sum().sort_values(ascending=False)
        for file, total in file_totals.items():
            print(f"{file:60s} : {total:10.2f} Mt")
        
        print("\n" + "="*80)
        print("诊断结论:")
        print("="*80)
        
        # 检查是否存在异常大的差异
        max_total = file_totals.max()
        min_total = file_totals.min()
        ratio = max_total / min_total if min_total > 0 else float('inf')
        
        print(f"最大潜力: {max_total:.2f} Mt")
        print(f"最小潜力: {min_total:.2f} Mt")
        print(f"比值: {ratio:.2f}x")
        
        if ratio > 10:
            print("\n⚠️  警告：不同年份的潜力差异超过10倍！")
            print("可能的原因:")
            print("1. 不同文件的潜力列使用了不同单位（kg vs ton vs Mt）")
            print("2. 数据预处理步骤不一致")
            print("3. 某些年份的数据过滤条件不同")
            print("4. 累积潜力的计算方式有误（是否重复累加？）")
            print("\n建议检查:")
            print("- 确保所有文件使用相同的单位")
            print("- 检查前面生成潜力数据的代码是否对不同年份使用了一致的逻辑")
            print("- 确认 production_potential 列的含义在所有文件中是否一致")
        else:
            print("\n✓ 不同年份的潜力差异在合理范围内")

## 查看2050年TOP10潜力点的完整数据

In [ ]:
import pandas as pd
import numpy as np

# 读取2050年数据文件
file_path = 'DataOutput/Used_output/combined_RegionRE_LowAE_2050_hydrogen_production_results_withPotential.csv'

print("="*100)
print("2050年氢能数据 - TOP 10 潜力点完整信息")
print("="*100)

df = pd.read_csv(file_path)

# 设置pandas显示选项，确保显示所有列和完整数值
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:.6e}' if abs(x) > 1e6 or (abs(x) < 1e-3 and x != 0) else f'{x:.6f}')

print(f"\n总数据点: {len(df)}")
print(f"总列数: {len(df.columns)}")
print(f"\n所有列名:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

# 获取 production_potential_ton 列
pot_col = 'production_potential_ton'
if pot_col not in df.columns:
    print(f"\n错误：未找到列 '{pot_col}'")
else:
    # 转换为数值并找出TOP10
    df['potential_numeric'] = pd.to_numeric(df[pot_col], errors='coerce')
    
    # 只考虑有效值
    df_valid = df[np.isfinite(df['potential_numeric']) & (df['potential_numeric'] > 0)].copy()
    
    print(f"\n有效潜力数据点: {len(df_valid)}")
    
    # 按潜力降序排序，取TOP10
    top10 = df_valid.nlargest(10, 'potential_numeric')
    
    print("\n" + "="*100)
    print("TOP 10 最大潜力点")
    print("="*100)
    
    # 逐行显示TOP10的完整信息
    for rank, (idx, row) in enumerate(top10.iterrows(), 1):
        print(f"\n{'='*100}")
        print(f"排名 #{rank} | 数据行索引: {idx}")
        print(f"{'='*100}")
        
        # 显示所有字段
        for col in df.columns:
            if col == 'potential_numeric':
                continue
            value = row[col]
            
            # 格式化显示
            if pd.isna(value):
                display_value = "NaN"
            elif isinstance(value, (int, np.integer)):
                display_value = f"{value:,}"
            elif isinstance(value, (float, np.floating)):
                if abs(value) > 1e6 or (abs(value) < 1e-3 and value != 0):
                    display_value = f"{value:.6e}"
                else:
                    display_value = f"{value:.6f}"
            else:
                display_value = str(value)
            
            print(f"  {col:50s} : {display_value}")
    
    # 统计分析
    print("\n" + "="*100)
    print("TOP 10 统计分析")
    print("="*100)
    
    top10_potential = top10['potential_numeric']
    total_potential = df_valid['potential_numeric'].sum()
    top10_sum = top10_potential.sum()
    
    print(f"\nTOP 10 潜力总和: {top10_sum:.6e} ton")
    print(f"所有有效点潜力总和: {total_potential:.6e} ton")
    print(f"TOP 10 占比: {top10_sum/total_potential*100:.2f}%")
    print(f"\nTOP 10 潜力范围:")
    print(f"  最大值 (Rank #1): {top10_potential.max():.6e} ton")
    print(f"  最小值 (Rank #10): {top10_potential.min():.6e} ton")
    print(f"  比值: {top10_potential.max() / top10_potential.min():.2f}x")
    
    # 检查是否有超大异常值
    print("\n" + "="*100)
    print("异常值检测")
    print("="*100)
    
    thresholds = [1e10, 1e12, 1e14, 1e16]
    for threshold in thresholds:
        count = (df_valid['potential_numeric'] > threshold).sum()
        if count > 0:
            sum_above = df_valid[df_valid['potential_numeric'] > threshold]['potential_numeric'].sum()
            pct = sum_above / total_potential * 100
            print(f"\n大于 {threshold:.0e} ton 的点:")
            print(f"  数量: {count}")
            print(f"  总和: {sum_above:.6e} ton")
            print(f"  占总潜力比例: {pct:.2f}%")
    
    # 对比其他列的值
    print("\n" + "="*100)
    print("TOP 10 的其他关键指标对比")
    print("="*100)
    
    key_cols = ['levelized_hydrogen_production_cost_per_kg', 'wind_capacity', 'pv_capacity', 
                'electrolyzer_capacity_AE', 'electrolyzer_capacity_PEM', 
                'electrolyzer_utilization_hours_percent', 'wind_potential', 'pv_potential']
    
    available_cols = [col for col in key_cols if col in top10.columns]
    
    if available_cols:
        print(f"\n{'Rank':<6} ", end='')
        for col in available_cols:
            print(f"{col[:20]:<22} ", end='')
        print()
        print("-" * 100)
        
        for rank, (idx, row) in enumerate(top10.iterrows(), 1):
            print(f"{rank:<6} ", end='')
            for col in available_cols:
                val = row[col]
                if pd.isna(val):
                    print(f"{'N/A':<22} ", end='')
                elif abs(val) > 1e6:
                    print(f"{val:.4e}  ", end='')
                else:
                    print(f"{val:<22.4f} ", end='')
            print()

print("\n" + "="*100)
print("分析完成！")
print("="*100)